# 17. OpenCV Telea versus LaMa Comparison

This notebook compares the OpenCV Telea and LaMa restoration baselines on the controlled 50-painting subset.

Both models are compared using the same:

- paintings,
- synthetic masks,
- damaged inputs,
- clean references,
- evaluation regions,
- metric families.

The purpose is to identify where each model performs better, where metrics disagree, and which cases require visual diagnostic inspection.

## Comparison strategy

The comparison is case-paired rather than only aggregate-based.

Each OpenCV Telea case is matched with the corresponding LaMa case using:

- `painting_id`,
- `mask_id`,
- `mask_type`.

The notebook compares:

- classical metrics,
- LPIPS perceptual metrics,
- CLIP and DINOv2 feature-space similarity metrics.

The main local comparison uses:

- masked-region classical metrics,
- mask-bounding-box LPIPS metrics,
- mask-bounding-box CLIP/DINOv2 metrics.

This keeps the comparison focused on the damaged region while preserving full-image and content-region summaries for context.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from restoration_eval.reporting import (
    dataframe_to_html_table,
    html_escape,
    image_block,
)

print("Project root:", PROJECT_ROOT)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
figures_dir = PROJECT_ROOT / paths_cfg["figures_dir"]
reports_dir = PROJECT_ROOT / paths_cfg["reports_dir"]

metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

processed_metadata_path = processed_metadata_dir / "metadata_processed_clean.csv"

opencv_metadata_path = processed_metadata_dir / "metadata_restored_opencv_telea.csv"
lama_metadata_path = processed_metadata_dir / "metadata_restored_lama.csv"

opencv_classical_metrics_path = metrics_dir / "classical_metrics_opencv_telea_50.csv"
lama_classical_metrics_path = metrics_dir / "classical_metrics_lama_50.csv"

opencv_lpips_metrics_path = metrics_dir / "lpips_metrics_opencv_telea_50.csv"
lama_lpips_metrics_path = metrics_dir / "lpips_metrics_lama_50.csv"

opencv_feature_metrics_path = metrics_dir / "feature_similarity_opencv_telea_50.csv"
lama_feature_metrics_path = metrics_dir / "feature_similarity_lama_50.csv"

opencv_error_manifest_path = metrics_dir / "error_map_manifest_all_opencv_telea_50.csv"
lama_error_manifest_path = metrics_dir / "error_map_manifest_all_lama_50.csv"

model_pairing_path = metrics_dir / "model_pairing_opencv_lama_50.csv"

comparison_classical_path = metrics_dir / "comparison_classical_opencv_lama_50.csv"
comparison_lpips_path = metrics_dir / "comparison_lpips_opencv_lama_50.csv"
comparison_feature_path = metrics_dir / "comparison_feature_similarity_opencv_lama_50.csv"

comparison_unified_path = metrics_dir / "comparison_unified_opencv_lama_50.csv"
comparison_summary_by_mask_type_path = metrics_dir / "comparison_summary_by_mask_type_opencv_lama_50.csv"
comparison_summary_by_category_path = metrics_dir / "comparison_summary_by_category_opencv_lama_50.csv"
comparison_win_rates_path = metrics_dir / "comparison_win_rates_opencv_lama_50.csv"
comparison_disagreement_cases_path = metrics_dir / "comparison_metric_disagreement_cases_opencv_lama_50.csv"
comparison_visual_cases_path = metrics_dir / "comparison_visual_cases_opencv_lama_50.csv"

comparison_figures_dir = figures_dir / "model_comparison" / "opencv_vs_lama"
comparison_figures_dir.mkdir(parents=True, exist_ok=True)

comparison_report_path = reports_dir / "opencv_vs_lama_comparison_report_50.html"

print("Processed metadata:", processed_metadata_path)
print("OpenCV metadata:", opencv_metadata_path)
print("LaMa metadata:", lama_metadata_path)
print("Comparison figures:", comparison_figures_dir)
print("Comparison report:", comparison_report_path)

Processed metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_processed_clean.csv
OpenCV metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_opencv_telea.csv
LaMa metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_lama.csv
Comparison figures: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\figures\model_comparison\opencv_vs_lama
Comparison report: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_vs_lama_comparison_report_50.html


In [3]:
required_input_paths = [
    processed_metadata_path,
    opencv_metadata_path,
    lama_metadata_path,
    opencv_classical_metrics_path,
    lama_classical_metrics_path,
    opencv_lpips_metrics_path,
    lama_lpips_metrics_path,
    opencv_feature_metrics_path,
    lama_feature_metrics_path,
    opencv_error_manifest_path,
    lama_error_manifest_path,
]

for input_path in required_input_paths:
    if not input_path.exists():
        raise FileNotFoundError(f"Missing required input file: {input_path}")

processed_metadata_df = pd.read_csv(processed_metadata_path)

opencv_metadata_df = pd.read_csv(opencv_metadata_path)
lama_metadata_df = pd.read_csv(lama_metadata_path)

opencv_classical_df = pd.read_csv(opencv_classical_metrics_path)
lama_classical_df = pd.read_csv(lama_classical_metrics_path)

opencv_lpips_df = pd.read_csv(opencv_lpips_metrics_path)
lama_lpips_df = pd.read_csv(lama_lpips_metrics_path)

opencv_feature_df = pd.read_csv(opencv_feature_metrics_path)
lama_feature_df = pd.read_csv(lama_feature_metrics_path)

opencv_error_manifest_df = pd.read_csv(opencv_error_manifest_path)
lama_error_manifest_df = pd.read_csv(lama_error_manifest_path)

print("Processed metadata shape:", processed_metadata_df.shape)
print("OpenCV metadata shape:", opencv_metadata_df.shape)
print("LaMa metadata shape:", lama_metadata_df.shape)

print("\nClassical metrics:")
print("OpenCV:", opencv_classical_df.shape)
print("LaMa:", lama_classical_df.shape)

print("\nLPIPS metrics:")
print("OpenCV:", opencv_lpips_df.shape)
print("LaMa:", lama_lpips_df.shape)

print("\nFeature metrics:")
print("OpenCV:", opencv_feature_df.shape)
print("LaMa:", lama_feature_df.shape)

print("\nError-map manifests:")
print("OpenCV:", opencv_error_manifest_df.shape)
print("LaMa:", lama_error_manifest_df.shape)

print("\nOpenCV model names:")
display(opencv_metadata_df["model_name"].value_counts(dropna=False))

print("\nLaMa model names:")
display(lama_metadata_df["model_name"].value_counts(dropna=False))

Processed metadata shape: (50, 43)
OpenCV metadata shape: (250, 20)
LaMa metadata shape: (250, 32)

Classical metrics:
OpenCV: (900, 30)
LaMa: (900, 30)

LPIPS metrics:
OpenCV: (700, 24)
LaMa: (700, 24)

Feature metrics:
OpenCV: (700, 28)
LaMa: (700, 28)

Error-map manifests:
OpenCV: (250, 29)
LaMa: (250, 29)

OpenCV model names:


model_name
opencv_telea    250
Name: count, dtype: int64


LaMa model names:


model_name
lama    250
Name: count, dtype: int64

In [4]:
expected_row_counts = {
    "processed_metadata": 50,
    "opencv_metadata": 250,
    "lama_metadata": 250,
    "opencv_classical": 900,
    "lama_classical": 900,
    "opencv_lpips": 700,
    "lama_lpips": 700,
    "opencv_feature": 700,
    "lama_feature": 700,
    "opencv_error_manifest": 250,
    "lama_error_manifest": 250,
}

actual_row_counts = {
    "processed_metadata": len(processed_metadata_df),
    "opencv_metadata": len(opencv_metadata_df),
    "lama_metadata": len(lama_metadata_df),
    "opencv_classical": len(opencv_classical_df),
    "lama_classical": len(lama_classical_df),
    "opencv_lpips": len(opencv_lpips_df),
    "lama_lpips": len(lama_lpips_df),
    "opencv_feature": len(opencv_feature_df),
    "lama_feature": len(lama_feature_df),
    "opencv_error_manifest": len(opencv_error_manifest_df),
    "lama_error_manifest": len(lama_error_manifest_df),
}

print("Expected row counts:")
display(pd.Series(expected_row_counts, name="expected"))

print("Actual row counts:")
display(pd.Series(actual_row_counts, name="actual"))

for name, expected_count in expected_row_counts.items():
    actual_count = actual_row_counts[name]
    if actual_count != expected_count:
        raise ValueError(
            f"{name}: expected {expected_count} rows, found {actual_count}."
        )

expected_models = {
    "opencv_metadata": "opencv_telea",
    "lama_metadata": "lama",
}

for dataframe_name, expected_model in expected_models.items():
    dataframe = {
        "opencv_metadata": opencv_metadata_df,
        "lama_metadata": lama_metadata_df,
    }[dataframe_name]

    actual_models = set(dataframe["model_name"].dropna().unique())

    print(f"{dataframe_name} models:", actual_models)

    if actual_models != {expected_model}:
        raise ValueError(
            f"{dataframe_name}: expected model {expected_model!r}, found {actual_models}."
        )

for dataframe_name, dataframe in [
    ("opencv_metadata_df", opencv_metadata_df),
    ("lama_metadata_df", lama_metadata_df),
    ("opencv_classical_df", opencv_classical_df),
    ("lama_classical_df", lama_classical_df),
    ("opencv_lpips_df", opencv_lpips_df),
    ("lama_lpips_df", lama_lpips_df),
    ("opencv_feature_df", opencv_feature_df),
    ("lama_feature_df", lama_feature_df),
    ("opencv_error_manifest_df", opencv_error_manifest_df),
    ("lama_error_manifest_df", lama_error_manifest_df),
]:
    if "status" in dataframe.columns:
        non_ok_count = int((dataframe["status"] != "ok").sum())
        print(f"{dataframe_name} non-ok rows:", non_ok_count)

        if non_ok_count != 0:
            display(dataframe[dataframe["status"] != "ok"].head(20))
            raise ValueError(f"{dataframe_name} contains non-ok rows.")

print("\nComparison input validation gates passed.")

Expected row counts:


processed_metadata        50
opencv_metadata          250
lama_metadata            250
opencv_classical         900
lama_classical           900
opencv_lpips             700
lama_lpips               700
opencv_feature           700
lama_feature             700
opencv_error_manifest    250
lama_error_manifest      250
Name: expected, dtype: int64

Actual row counts:


processed_metadata        50
opencv_metadata          250
lama_metadata            250
opencv_classical         900
lama_classical           900
opencv_lpips             700
lama_lpips               700
opencv_feature           700
lama_feature             700
opencv_error_manifest    250
lama_error_manifest      250
Name: actual, dtype: int64

opencv_metadata models: {'opencv_telea'}
lama_metadata models: {'lama'}
opencv_metadata_df non-ok rows: 0
lama_metadata_df non-ok rows: 0
opencv_classical_df non-ok rows: 0
lama_classical_df non-ok rows: 0
opencv_lpips_df non-ok rows: 0
lama_lpips_df non-ok rows: 0
opencv_feature_df non-ok rows: 0
lama_feature_df non-ok rows: 0
opencv_error_manifest_df non-ok rows: 0
lama_error_manifest_df non-ok rows: 0

Comparison input validation gates passed.


In [5]:
expected_classical_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "masked_region": 200,
    "mask_bbox_crop": 200,
}

expected_spatial_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

for dataframe_name, dataframe in [
    ("opencv_classical_df", opencv_classical_df),
    ("lama_classical_df", lama_classical_df),
]:
    actual_counts = dataframe["evaluation_region"].value_counts().to_dict()

    print(f"\n{dataframe_name} region counts:")
    print(actual_counts)

    for region, expected_count in expected_classical_region_counts.items():
        actual_count = actual_counts.get(region, 0)
        if actual_count != expected_count:
            raise ValueError(
                f"{dataframe_name} region {region!r}: "
                f"expected {expected_count}, found {actual_count}."
            )

for dataframe_name, dataframe in [
    ("opencv_lpips_df", opencv_lpips_df),
    ("lama_lpips_df", lama_lpips_df),
    ("opencv_feature_df", opencv_feature_df),
    ("lama_feature_df", lama_feature_df),
]:
    actual_counts = dataframe["evaluation_region"].value_counts().to_dict()

    print(f"\n{dataframe_name} region counts:")
    print(actual_counts)

    for region, expected_count in expected_spatial_region_counts.items():
        actual_count = actual_counts.get(region, 0)
        if actual_count != expected_count:
            raise ValueError(
                f"{dataframe_name} region {region!r}: "
                f"expected {expected_count}, found {actual_count}."
            )

print("\nMetric region-count validation passed.")


opencv_classical_df region counts:
{'full_image': 250, 'content_region': 250, 'masked_region': 200, 'mask_bbox_crop': 200}

lama_classical_df region counts:
{'full_image': 250, 'content_region': 250, 'masked_region': 200, 'mask_bbox_crop': 200}

opencv_lpips_df region counts:
{'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

lama_lpips_df region counts:
{'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

opencv_feature_df region counts:
{'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

lama_feature_df region counts:
{'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

Metric region-count validation passed.


In [6]:
pairing_keys = ["painting_id", "mask_id", "mask_type"]

opencv_pairing_columns = pairing_keys + [
    "case_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "status",
]

lama_pairing_columns = pairing_keys + [
    "case_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "status",
]

opencv_pairing_df = opencv_metadata_df[opencv_pairing_columns].copy()
lama_pairing_df = lama_metadata_df[lama_pairing_columns].copy()

model_pairing_df = opencv_pairing_df.merge(
    lama_pairing_df,
    on=pairing_keys,
    how="inner",
    suffixes=("_opencv", "_lama"),
    validate="one_to_one",
)

print("Model pairing shape:", model_pairing_df.shape)
display(model_pairing_df.head())

if len(model_pairing_df) != 250:
    raise ValueError(
        f"Expected 250 paired OpenCV-LaMa cases, found {len(model_pairing_df)}."
    )

if model_pairing_df[pairing_keys].duplicated().any():
    duplicated_rows = model_pairing_df[model_pairing_df[pairing_keys].duplicated(keep=False)]
    display(duplicated_rows)
    raise ValueError("Model pairing contains duplicate pairing keys.")

if (model_pairing_df["status_opencv"] != "ok").any():
    raise ValueError("OpenCV paired metadata contains non-ok rows.")

if (model_pairing_df["status_lama"] != "ok").any():
    raise ValueError("LaMa paired metadata contains non-ok rows.")

if set(model_pairing_df["model_name_opencv"].dropna().unique()) != {"opencv_telea"}:
    raise ValueError(
        f"Unexpected OpenCV model names: {model_pairing_df['model_name_opencv'].unique()}"
    )

if set(model_pairing_df["model_name_lama"].dropna().unique()) != {"lama"}:
    raise ValueError(
        f"Unexpected LaMa model names: {model_pairing_df['model_name_lama'].unique()}"
    )

path_consistency_columns = [
    ("clean_path_opencv", "clean_path_lama"),
    ("damaged_path_opencv", "damaged_path_lama"),
    ("mask_path_opencv", "mask_path_lama"),
]

for opencv_column, lama_column in path_consistency_columns:
    mismatch_count = int(
        (model_pairing_df[opencv_column].astype(str) != model_pairing_df[lama_column].astype(str)).sum()
    )

    print(f"{opencv_column} vs {lama_column} mismatches:", mismatch_count)

    if mismatch_count != 0:
        mismatch_df = model_pairing_df[
            model_pairing_df[opencv_column].astype(str)
            != model_pairing_df[lama_column].astype(str)
        ]
        display(mismatch_df[[*pairing_keys, opencv_column, lama_column]].head(20))
        raise ValueError(
            f"Paired models do not share identical paths for {opencv_column} and {lama_column}."
        )

model_pairing_df = model_pairing_df.merge(
    processed_metadata_df[
        [
            "painting_id",
            "title",
            "artist",
            "date",
            "category",
            "style_or_period",
            "medium",
            "source",
            "source_url",
            "license",
            "filename",
        ]
    ],
    on="painting_id",
    how="left",
    validate="many_to_one",
)

if model_pairing_df["category"].isna().any():
    raise ValueError("Model pairing contains missing category values after metadata merge.")

model_pairing_df.to_csv(model_pairing_path, index=False)

print("\nSaved model-pairing table:")
print(model_pairing_path)

print("\nPairing mask counts:")
display(model_pairing_df["mask_type"].value_counts().sort_index())

print("\nPairing category counts:")
display(model_pairing_df["category"].value_counts().sort_index())

Model pairing shape: (250, 17)


,painting_id,mask_id,mask_type,case_id_opencv,model_name_opencv,clean_path_opencv,damaged_path_opencv,restored_path_opencv,mask_path_opencv,status_opencv,case_id_lama,model_name_lama,clean_path_lama,damaged_path_lama,restored_path_lama,mask_path_lama,status_lama
0,p001,p001_loss_large,loss_large,p001_loss_large,opencv_telea,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok,p001_loss_large,lama,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p001_loss_large_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok
1,p001,p001_loss_small,loss_small,p001_loss_small,opencv_telea,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok,p001_loss_small,lama,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p001_loss_small_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok
2,p001,p001_mixed_damage,mixed_damage,p001_mixed_damage,opencv_telea,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok,p001_mixed_damage,lama,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p001_mixed_damage...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok
3,p001,p001_scratch_thin,scratch_thin,p001_scratch_thin,opencv_telea,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok,p001_scratch_thin,lama,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p001_scratch_thin...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok
4,p001,p001_zero_control,zero_control,p001_zero_control,opencv_telea,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok,p001_zero_control,lama,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p001_zero_control...,D:\Masters\FH\Thesis\painting-restoration-eval...,ok


clean_path_opencv vs clean_path_lama mismatches: 0
damaged_path_opencv vs damaged_path_lama mismatches: 0
mask_path_opencv vs mask_path_lama mismatches: 0

Saved model-pairing table:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\model_pairing_opencv_lama_50.csv

Pairing mask counts:


mask_type
loss_large      50
loss_small      50
mixed_damage    50
scratch_thin    50
zero_control    50
Name: count, dtype: int64


Pairing category counts:


category
abstraction_surrealism     50
architecture_structured    50
high_texture_brushwork     50
landscape_natural          50
portrait_figure            50
Name: count, dtype: int64

In [7]:
def build_metric_comparison(
    *,
    opencv_df: pd.DataFrame,
    lama_df: pd.DataFrame,
    metric_family: str,
    metric_columns: list[str],
    pairing_keys: list[str],
    extra_keys: list[str] | None = None,
) -> pd.DataFrame:
    """Pair OpenCV and LaMa metric rows and compute LaMa-minus-OpenCV deltas."""
    if extra_keys is None:
        extra_keys = []

    join_keys = pairing_keys + extra_keys

    required_columns = join_keys + metric_columns

    for dataframe_name, dataframe in [
        ("opencv_df", opencv_df),
        ("lama_df", lama_df),
    ]:
        missing_columns = [
            column for column in required_columns
            if column not in dataframe.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{dataframe_name} missing columns for {metric_family}: {missing_columns}"
            )

    opencv_metric_df = opencv_df[required_columns].copy()
    lama_metric_df = lama_df[required_columns].copy()

    comparison_df = opencv_metric_df.merge(
        lama_metric_df,
        on=join_keys,
        how="inner",
        suffixes=("_opencv", "_lama"),
        validate="one_to_one",
    )

    for metric_column in metric_columns:
        comparison_df[f"{metric_column}_delta_lama_minus_opencv"] = (
            comparison_df[f"{metric_column}_lama"]
            - comparison_df[f"{metric_column}_opencv"]
        )

    comparison_df["metric_family"] = metric_family

    return comparison_df


def winner_from_delta(
    delta: float,
    *,
    higher_is_better: bool,
    tolerance: float = 1e-12,
) -> str:
    """Return the winning model from a LaMa-minus-OpenCV delta."""
    if pd.isna(delta):
        return "missing"

    if abs(delta) <= tolerance:
        return "tie"

    if higher_is_better:
        return "lama" if delta > 0 else "opencv"

    return "lama" if delta < 0 else "opencv"


def add_winner_column(
    df: pd.DataFrame,
    *,
    delta_column: str,
    winner_column: str,
    higher_is_better: bool,
) -> pd.DataFrame:
    """Add a winner column using a LaMa-minus-OpenCV delta column."""
    output_df = df.copy()

    output_df[winner_column] = output_df[delta_column].map(
        lambda value: winner_from_delta(
            value,
            higher_is_better=higher_is_better,
        )
    )

    return output_df


print("Metric comparison helpers ready.")

Metric comparison helpers ready.


In [8]:
classical_pairing_keys = ["case_id", "painting_id", "mask_id", "mask_type"]
classical_extra_keys = ["evaluation_region"]

classical_metric_columns = [
    "damaged_mse",
    "restored_mse",
    "mse_improvement",
    "damaged_mae",
    "restored_mae",
    "mae_improvement",
    "damaged_psnr",
    "restored_psnr",
    "psnr_improvement",
]

optional_classical_columns = [
    "damaged_ssim",
    "restored_ssim",
    "ssim_improvement",
]

for optional_column in optional_classical_columns:
    if optional_column in opencv_classical_df.columns and optional_column in lama_classical_df.columns:
        classical_metric_columns.append(optional_column)

comparison_classical_df = build_metric_comparison(
    opencv_df=opencv_classical_df,
    lama_df=lama_classical_df,
    metric_family="classical",
    metric_columns=classical_metric_columns,
    pairing_keys=classical_pairing_keys,
    extra_keys=classical_extra_keys,
)

comparison_classical_df = comparison_classical_df.merge(
    processed_metadata_df[
        [
            "painting_id",
            "title",
            "artist",
            "date",
            "category",
            "style_or_period",
            "medium",
            "source",
            "source_url",
            "license",
            "filename",
        ]
    ],
    on="painting_id",
    how="left",
    validate="many_to_one",
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="restored_mse_delta_lama_minus_opencv",
    winner_column="winner_restored_mse",
    higher_is_better=False,
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="mse_improvement_delta_lama_minus_opencv",
    winner_column="winner_mse_improvement",
    higher_is_better=True,
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="restored_mae_delta_lama_minus_opencv",
    winner_column="winner_restored_mae",
    higher_is_better=False,
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="mae_improvement_delta_lama_minus_opencv",
    winner_column="winner_mae_improvement",
    higher_is_better=True,
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="restored_psnr_delta_lama_minus_opencv",
    winner_column="winner_restored_psnr",
    higher_is_better=True,
)

comparison_classical_df = add_winner_column(
    comparison_classical_df,
    delta_column="psnr_improvement_delta_lama_minus_opencv",
    winner_column="winner_psnr_improvement",
    higher_is_better=True,
)

if "restored_ssim_delta_lama_minus_opencv" in comparison_classical_df.columns:
    comparison_classical_df = add_winner_column(
        comparison_classical_df,
        delta_column="restored_ssim_delta_lama_minus_opencv",
        winner_column="winner_restored_ssim",
        higher_is_better=True,
    )

if "ssim_improvement_delta_lama_minus_opencv" in comparison_classical_df.columns:
    comparison_classical_df = add_winner_column(
        comparison_classical_df,
        delta_column="ssim_improvement_delta_lama_minus_opencv",
        winner_column="winner_ssim_improvement",
        higher_is_better=True,
    )

print("Classical comparison shape:", comparison_classical_df.shape)
display(comparison_classical_df.head())

print("\nClassical comparison region counts:")
display(comparison_classical_df["evaluation_region"].value_counts())

print("\nMasked-region MSE improvement winners:")
display(
    comparison_classical_df[
        comparison_classical_df["evaluation_region"] == "masked_region"
    ]["winner_mse_improvement"].value_counts(dropna=False)
)

Classical comparison shape: (900, 60)


,case_id,painting_id,mask_id,mask_type,evaluation_region,damaged_mse_opencv,restored_mse_opencv,mse_improvement_opencv,damaged_mae_opencv,restored_mae_opencv,...,license,filename,winner_restored_mse,winner_mse_improvement,winner_restored_mae,winner_mae_improvement,winner_restored_psnr,winner_psnr_improvement,winner_restored_ssim,winner_ssim_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,full_image,5585.921875,111.934753,5473.987122,25.058495,2.166397,...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama,lama,lama,lama,lama
1,p001_loss_large,p001,p001_loss_large,loss_large,content_region,6470.570312,129.661972,6340.908340,29.027035,2.509491,...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama,lama,lama,lama,lama
2,p001_loss_large,p001,p001_loss_large,loss_large,masked_region,48997.804688,981.853638,48015.951050,219.804611,19.002895,...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama,lama,lama,missing,missing
3,p001_loss_large,p001,p001_loss_large,loss_large,mask_bbox_crop,25993.773438,520.882080,25472.891357,116.608292,10.081204,...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama,lama,lama,lama,lama
4,p001_loss_small,p001,p001_loss_small,loss_small,full_image,1928.027954,1.341102,1926.686852,8.864422,0.145845,...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama,lama,lama,opencv,opencv



Classical comparison region counts:


evaluation_region
full_image        250
content_region    250
masked_region     200
mask_bbox_crop    200
Name: count, dtype: int64


Masked-region MSE improvement winners:


winner_mse_improvement
lama      159
opencv     41
Name: count, dtype: int64

In [13]:
psnr_delta_pairs = [
    (
        "restored_psnr_opencv",
        "restored_psnr_lama",
        "restored_psnr_delta_lama_minus_opencv",
        "winner_restored_psnr",
    ),
    (
        "psnr_improvement_opencv",
        "psnr_improvement_lama",
        "psnr_improvement_delta_lama_minus_opencv",
        "winner_psnr_improvement",
    ),
]

for opencv_column, lama_column, delta_column, winner_column in psnr_delta_pairs:
    if delta_column not in comparison_classical_df.columns:
        continue

    both_infinite_mask = (
        np.isinf(comparison_classical_df[opencv_column])
        & np.isinf(comparison_classical_df[lama_column])
        & comparison_classical_df[delta_column].isna()
    )

    both_missing_zero_control_mask = (
        comparison_classical_df["mask_type"].eq("zero_control")
        & comparison_classical_df[opencv_column].isna()
        & comparison_classical_df[lama_column].isna()
        & comparison_classical_df[delta_column].isna()
    )

    fix_mask = both_infinite_mask | both_missing_zero_control_mask

    print(delta_column, "rows fixed:", int(fix_mask.sum()))

    comparison_classical_df.loc[fix_mask, delta_column] = 0.0
    comparison_classical_df.loc[fix_mask, winner_column] = "tie"

remaining_psnr_delta_na = comparison_classical_df[
    [
        "restored_psnr_delta_lama_minus_opencv",
        "psnr_improvement_delta_lama_minus_opencv",
    ]
].isna().sum()

print("\nRemaining PSNR delta NaNs:")
display(remaining_psnr_delta_na)

restored_psnr_delta_lama_minus_opencv rows fixed: 0
psnr_improvement_delta_lama_minus_opencv rows fixed: 100

Remaining PSNR delta NaNs:


restored_psnr_delta_lama_minus_opencv       0
psnr_improvement_delta_lama_minus_opencv    0
dtype: int64

In [14]:
if len(comparison_classical_df) != 900:
    raise ValueError(
        f"Expected 900 classical comparison rows, found {len(comparison_classical_df)}."
    )

expected_classical_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "masked_region": 200,
    "mask_bbox_crop": 200,
}

actual_classical_region_counts = comparison_classical_df["evaluation_region"].value_counts().to_dict()

for region, expected_count in expected_classical_region_counts.items():
    actual_count = actual_classical_region_counts.get(region, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Classical comparison region {region!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

required_classical_delta_columns = [
    "restored_mse_delta_lama_minus_opencv",
    "mse_improvement_delta_lama_minus_opencv",
    "restored_mae_delta_lama_minus_opencv",
    "mae_improvement_delta_lama_minus_opencv",
    "restored_psnr_delta_lama_minus_opencv",
    "psnr_improvement_delta_lama_minus_opencv",
]

for delta_column in required_classical_delta_columns:
    missing_count = int(comparison_classical_df[delta_column].isna().sum())

    if missing_count != 0:
        display(
            comparison_classical_df[
                comparison_classical_df[delta_column].isna()
            ][
                [
                    "case_id",
                    "painting_id",
                    "mask_id",
                    "mask_type",
                    "evaluation_region",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_opencv",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_lama",
                    delta_column,
                ]
            ].head(20)
        )

        raise ValueError(
            f"Missing values found in {delta_column!r}: {missing_count} rows."
        )

if comparison_classical_df["category"].isna().any():
    raise ValueError("Classical comparison contains missing category labels.")

comparison_classical_df.to_csv(comparison_classical_path, index=False)

print("Saved classical comparison:")
print(comparison_classical_path)

print("\nClassical comparison validation passed.")

Saved classical comparison:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_classical_opencv_lama_50.csv

Classical comparison validation passed.


In [15]:
lpips_pairing_keys = ["case_id", "painting_id", "mask_id", "mask_type"]
lpips_extra_keys = ["evaluation_region"]

lpips_metric_columns = [
    "damaged_lpips",
    "restored_lpips",
    "lpips_improvement",
]

comparison_lpips_df = build_metric_comparison(
    opencv_df=opencv_lpips_df,
    lama_df=lama_lpips_df,
    metric_family="lpips",
    metric_columns=lpips_metric_columns,
    pairing_keys=lpips_pairing_keys,
    extra_keys=lpips_extra_keys,
)

comparison_lpips_df = comparison_lpips_df.merge(
    processed_metadata_df[
        [
            "painting_id",
            "title",
            "artist",
            "date",
            "category",
            "style_or_period",
            "medium",
            "source",
            "source_url",
            "license",
            "filename",
        ]
    ],
    on="painting_id",
    how="left",
    validate="many_to_one",
)

comparison_lpips_df = add_winner_column(
    comparison_lpips_df,
    delta_column="restored_lpips_delta_lama_minus_opencv",
    winner_column="winner_restored_lpips",
    higher_is_better=False,
)

comparison_lpips_df = add_winner_column(
    comparison_lpips_df,
    delta_column="lpips_improvement_delta_lama_minus_opencv",
    winner_column="winner_lpips_improvement",
    higher_is_better=True,
)

print("LPIPS comparison shape:", comparison_lpips_df.shape)
display(comparison_lpips_df.head())

print("\nLPIPS comparison region counts:")
display(comparison_lpips_df["evaluation_region"].value_counts())

print("\nMask-bbox LPIPS improvement winners:")
display(
    comparison_lpips_df[
        comparison_lpips_df["evaluation_region"] == "mask_bbox_crop"
    ]["winner_lpips_improvement"].value_counts(dropna=False)
)

LPIPS comparison shape: (700, 27)


,case_id,painting_id,mask_id,mask_type,evaluation_region,damaged_lpips_opencv,restored_lpips_opencv,lpips_improvement_opencv,damaged_lpips_lama,restored_lpips_lama,...,date,category,style_or_period,medium,source,source_url,license,filename,winner_restored_lpips,winner_lpips_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,full_image,0.264050,0.088961,0.175089,0.264050,0.045897,...,1650,portrait_figure,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama
1,p001_loss_large,p001,p001_loss_large,loss_large,content_region,0.288218,0.101456,0.186762,0.288218,0.049712,...,1650,portrait_figure,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama
2,p001_loss_large,p001,p001_loss_large,loss_large,mask_bbox_crop,0.592107,0.273881,0.318226,0.592107,0.147469,...,1650,portrait_figure,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama
3,p001_loss_small,p001,p001_loss_small,loss_small,full_image,0.335288,0.005583,0.329705,0.335288,0.002392,...,1650,portrait_figure,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama
4,p001_loss_small,p001,p001_loss_small,loss_small,content_region,0.318926,0.006440,0.312486,0.318926,0.002862,...,1650,portrait_figure,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama



LPIPS comparison region counts:


evaluation_region
full_image        250
content_region    250
mask_bbox_crop    200
Name: count, dtype: int64


Mask-bbox LPIPS improvement winners:


winner_lpips_improvement
lama      188
opencv     12
Name: count, dtype: int64

In [16]:
if len(comparison_lpips_df) != 700:
    raise ValueError(
        f"Expected 700 LPIPS comparison rows, found {len(comparison_lpips_df)}."
    )

expected_lpips_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

actual_lpips_region_counts = comparison_lpips_df["evaluation_region"].value_counts().to_dict()

for region, expected_count in expected_lpips_region_counts.items():
    actual_count = actual_lpips_region_counts.get(region, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"LPIPS comparison region {region!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

required_lpips_delta_columns = [
    "restored_lpips_delta_lama_minus_opencv",
    "lpips_improvement_delta_lama_minus_opencv",
]

for delta_column in required_lpips_delta_columns:
    missing_count = int(comparison_lpips_df[delta_column].isna().sum())

    if missing_count != 0:
        display(
            comparison_lpips_df[
                comparison_lpips_df[delta_column].isna()
            ][
                [
                    "case_id",
                    "painting_id",
                    "mask_id",
                    "mask_type",
                    "evaluation_region",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_opencv",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_lama",
                    delta_column,
                ]
            ].head(20)
        )

        raise ValueError(
            f"Missing values found in {delta_column!r}: {missing_count} rows."
        )

if comparison_lpips_df["category"].isna().any():
    raise ValueError("LPIPS comparison contains missing category labels.")

comparison_lpips_df.to_csv(comparison_lpips_path, index=False)

print("Saved LPIPS comparison:")
print(comparison_lpips_path)

print("\nLPIPS comparison validation passed.")

Saved LPIPS comparison:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_lpips_opencv_lama_50.csv

LPIPS comparison validation passed.


In [17]:
feature_pairing_keys = ["case_id", "painting_id", "mask_id", "mask_type"]
feature_extra_keys = ["evaluation_region"]

feature_metric_columns = [
    "clip_damaged_similarity",
    "clip_restored_similarity",
    "clip_similarity_improvement",
    "dinov2_damaged_similarity",
    "dinov2_restored_similarity",
    "dinov2_similarity_improvement",
]

comparison_feature_df = build_metric_comparison(
    opencv_df=opencv_feature_df,
    lama_df=lama_feature_df,
    metric_family="feature_similarity",
    metric_columns=feature_metric_columns,
    pairing_keys=feature_pairing_keys,
    extra_keys=feature_extra_keys,
)

comparison_feature_df = comparison_feature_df.merge(
    processed_metadata_df[
        [
            "painting_id",
            "title",
            "artist",
            "date",
            "category",
            "style_or_period",
            "medium",
            "source",
            "source_url",
            "license",
            "filename",
        ]
    ],
    on="painting_id",
    how="left",
    validate="many_to_one",
)

comparison_feature_df = add_winner_column(
    comparison_feature_df,
    delta_column="clip_restored_similarity_delta_lama_minus_opencv",
    winner_column="winner_clip_restored_similarity",
    higher_is_better=True,
)

comparison_feature_df = add_winner_column(
    comparison_feature_df,
    delta_column="clip_similarity_improvement_delta_lama_minus_opencv",
    winner_column="winner_clip_similarity_improvement",
    higher_is_better=True,
)

comparison_feature_df = add_winner_column(
    comparison_feature_df,
    delta_column="dinov2_restored_similarity_delta_lama_minus_opencv",
    winner_column="winner_dinov2_restored_similarity",
    higher_is_better=True,
)

comparison_feature_df = add_winner_column(
    comparison_feature_df,
    delta_column="dinov2_similarity_improvement_delta_lama_minus_opencv",
    winner_column="winner_dinov2_similarity_improvement",
    higher_is_better=True,
)

print("Feature-similarity comparison shape:", comparison_feature_df.shape)
display(comparison_feature_df.head())

print("\nFeature comparison region counts:")
display(comparison_feature_df["evaluation_region"].value_counts())

print("\nMask-bbox DINOv2 improvement winners:")
display(
    comparison_feature_df[
        comparison_feature_df["evaluation_region"] == "mask_bbox_crop"
    ]["winner_dinov2_similarity_improvement"].value_counts(dropna=False)
)

print("\nMask-bbox CLIP improvement winners:")
display(
    comparison_feature_df[
        comparison_feature_df["evaluation_region"] == "mask_bbox_crop"
    ]["winner_clip_similarity_improvement"].value_counts(dropna=False)
)

Feature-similarity comparison shape: (700, 38)


,case_id,painting_id,mask_id,mask_type,evaluation_region,clip_damaged_similarity_opencv,clip_restored_similarity_opencv,clip_similarity_improvement_opencv,dinov2_damaged_similarity_opencv,dinov2_restored_similarity_opencv,...,style_or_period,medium,source,source_url,license,filename,winner_clip_restored_similarity,winner_clip_similarity_improvement,winner_dinov2_restored_similarity,winner_dinov2_similarity_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,full_image,0.932040,0.967289,0.035249,0.796219,0.938681,...,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama
1,p001_loss_large,p001,p001_loss_large,loss_large,content_region,0.916990,0.964814,0.047824,0.790295,0.920064,...,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama
2,p001_loss_large,p001,p001_loss_large,loss_large,mask_bbox_crop,0.810900,0.808008,-0.002892,0.706465,0.748291,...,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama
3,p001_loss_small,p001,p001_loss_small,loss_small,full_image,0.914921,0.999334,0.084412,0.935493,0.998409,...,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama
4,p001_loss_small,p001,p001_loss_small,loss_small,content_region,0.920872,0.999257,0.078385,0.932966,0.998013,...,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,p001.jpg,lama,lama,lama,lama



Feature comparison region counts:


evaluation_region
full_image        250
content_region    250
mask_bbox_crop    200
Name: count, dtype: int64


Mask-bbox DINOv2 improvement winners:


winner_dinov2_similarity_improvement
lama      173
opencv     27
Name: count, dtype: int64


Mask-bbox CLIP improvement winners:


winner_clip_similarity_improvement
lama      175
opencv     25
Name: count, dtype: int64

In [18]:
if len(comparison_feature_df) != 700:
    raise ValueError(
        f"Expected 700 feature-similarity comparison rows, found {len(comparison_feature_df)}."
    )

expected_feature_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

actual_feature_region_counts = comparison_feature_df["evaluation_region"].value_counts().to_dict()

for region, expected_count in expected_feature_region_counts.items():
    actual_count = actual_feature_region_counts.get(region, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Feature comparison region {region!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

required_feature_delta_columns = [
    "clip_restored_similarity_delta_lama_minus_opencv",
    "clip_similarity_improvement_delta_lama_minus_opencv",
    "dinov2_restored_similarity_delta_lama_minus_opencv",
    "dinov2_similarity_improvement_delta_lama_minus_opencv",
]

for delta_column in required_feature_delta_columns:
    missing_count = int(comparison_feature_df[delta_column].isna().sum())

    if missing_count != 0:
        display(
            comparison_feature_df[
                comparison_feature_df[delta_column].isna()
            ][
                [
                    "case_id",
                    "painting_id",
                    "mask_id",
                    "mask_type",
                    "evaluation_region",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_opencv",
                    f"{delta_column.replace('_delta_lama_minus_opencv', '')}_lama",
                    delta_column,
                ]
            ].head(20)
        )

        raise ValueError(
            f"Missing values found in {delta_column!r}: {missing_count} rows."
        )

if comparison_feature_df["category"].isna().any():
    raise ValueError("Feature-similarity comparison contains missing category labels.")

comparison_feature_df.to_csv(comparison_feature_path, index=False)

print("Saved feature-similarity comparison:")
print(comparison_feature_path)

print("\nFeature-similarity comparison validation passed.")

Saved feature-similarity comparison:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_feature_similarity_opencv_lama_50.csv

Feature-similarity comparison validation passed.


In [19]:
# Local-region slices used for the main model comparison.
classical_local_df = comparison_classical_df[
    comparison_classical_df["evaluation_region"] == "masked_region"
].copy()

lpips_local_df = comparison_lpips_df[
    comparison_lpips_df["evaluation_region"] == "mask_bbox_crop"
].copy()

feature_local_df = comparison_feature_df[
    comparison_feature_df["evaluation_region"] == "mask_bbox_crop"
].copy()

print("Classical local rows:", len(classical_local_df))
print("LPIPS local rows:", len(lpips_local_df))
print("Feature local rows:", len(feature_local_df))

for name, dataframe in [
    ("classical_local_df", classical_local_df),
    ("lpips_local_df", lpips_local_df),
    ("feature_local_df", feature_local_df),
]:
    if len(dataframe) != 200:
        raise ValueError(f"{name}: expected 200 rows, found {len(dataframe)}.")

unified_keys = ["case_id", "painting_id", "mask_id", "mask_type"]

classical_unified_columns = unified_keys + [
    "category",
    "title",
    "artist",
    "date",
    "style_or_period",
    "medium",
    "source",
    "source_url",
    "license",
    "filename",
    "damaged_mse_opencv",
    "damaged_mse_lama",
    "restored_mse_opencv",
    "restored_mse_lama",
    "restored_mse_delta_lama_minus_opencv",
    "mse_improvement_opencv",
    "mse_improvement_lama",
    "mse_improvement_delta_lama_minus_opencv",
    "winner_mse_improvement",
    "damaged_mae_opencv",
    "damaged_mae_lama",
    "restored_mae_opencv",
    "restored_mae_lama",
    "restored_mae_delta_lama_minus_opencv",
    "mae_improvement_opencv",
    "mae_improvement_lama",
    "mae_improvement_delta_lama_minus_opencv",
    "winner_mae_improvement",
    "damaged_psnr_opencv",
    "damaged_psnr_lama",
    "restored_psnr_opencv",
    "restored_psnr_lama",
    "restored_psnr_delta_lama_minus_opencv",
    "psnr_improvement_opencv",
    "psnr_improvement_lama",
    "psnr_improvement_delta_lama_minus_opencv",
    "winner_psnr_improvement",
]

lpips_unified_columns = unified_keys + [
    "damaged_lpips_opencv",
    "damaged_lpips_lama",
    "restored_lpips_opencv",
    "restored_lpips_lama",
    "restored_lpips_delta_lama_minus_opencv",
    "lpips_improvement_opencv",
    "lpips_improvement_lama",
    "lpips_improvement_delta_lama_minus_opencv",
    "winner_lpips_improvement",
]

feature_unified_columns = unified_keys + [
    "clip_damaged_similarity_opencv",
    "clip_damaged_similarity_lama",
    "clip_restored_similarity_opencv",
    "clip_restored_similarity_lama",
    "clip_restored_similarity_delta_lama_minus_opencv",
    "clip_similarity_improvement_opencv",
    "clip_similarity_improvement_lama",
    "clip_similarity_improvement_delta_lama_minus_opencv",
    "winner_clip_similarity_improvement",
    "dinov2_damaged_similarity_opencv",
    "dinov2_damaged_similarity_lama",
    "dinov2_restored_similarity_opencv",
    "dinov2_restored_similarity_lama",
    "dinov2_restored_similarity_delta_lama_minus_opencv",
    "dinov2_similarity_improvement_opencv",
    "dinov2_similarity_improvement_lama",
    "dinov2_similarity_improvement_delta_lama_minus_opencv",
    "winner_dinov2_similarity_improvement",
]

missing_classical_columns = [
    column for column in classical_unified_columns
    if column not in classical_local_df.columns
]

missing_lpips_columns = [
    column for column in lpips_unified_columns
    if column not in lpips_local_df.columns
]

missing_feature_columns = [
    column for column in feature_unified_columns
    if column not in feature_local_df.columns
]

if missing_classical_columns:
    raise ValueError(f"Missing classical unified columns: {missing_classical_columns}")

if missing_lpips_columns:
    raise ValueError(f"Missing LPIPS unified columns: {missing_lpips_columns}")

if missing_feature_columns:
    raise ValueError(f"Missing feature unified columns: {missing_feature_columns}")

comparison_unified_df = classical_local_df[classical_unified_columns].merge(
    lpips_local_df[lpips_unified_columns],
    on=unified_keys,
    how="inner",
    validate="one_to_one",
)

comparison_unified_df = comparison_unified_df.merge(
    feature_local_df[feature_unified_columns],
    on=unified_keys,
    how="inner",
    validate="one_to_one",
)

print("Unified comparison shape:", comparison_unified_df.shape)
display(comparison_unified_df.head())

Classical local rows: 200
LPIPS local rows: 200
Feature local rows: 200
Unified comparison shape: (200, 68)


,case_id,painting_id,mask_id,mask_type,category,title,artist,date,style_or_period,medium,...,winner_clip_similarity_improvement,dinov2_damaged_similarity_opencv,dinov2_damaged_similarity_lama,dinov2_restored_similarity_opencv,dinov2_restored_similarity_lama,dinov2_restored_similarity_delta_lama_minus_opencv,dinov2_similarity_improvement_opencv,dinov2_similarity_improvement_lama,dinov2_similarity_improvement_delta_lama_minus_opencv,winner_dinov2_similarity_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,Juan de Pareja,Diego Velázquez,1650,Baroque,Oil on canvas,...,lama,0.706465,0.706465,0.748291,0.960092,0.211801,0.041827,0.253627,0.211801,lama
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,Juan de Pareja,Diego Velázquez,1650,Baroque,Oil on canvas,...,lama,0.948942,0.948942,0.997888,0.999632,0.001744,0.048946,0.050690,0.001744,lama
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,portrait_figure,Juan de Pareja,Diego Velázquez,1650,Baroque,Oil on canvas,...,lama,0.795198,0.795198,0.974854,0.992431,0.017577,0.179656,0.197233,0.017577,lama
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,portrait_figure,Juan de Pareja,Diego Velázquez,1650,Baroque,Oil on canvas,...,lama,0.924942,0.924942,0.998968,0.999538,0.000570,0.074026,0.074596,0.000570,lama
4,p002_loss_large,p002,p002_loss_large,loss_large,portrait_figure,Madame X (Madame Pierre Gautreau),John Singer Sargent,1883-84,19th century portraiture,Oil on canvas,...,lama,0.762878,0.762878,0.774365,0.935186,0.160821,0.011488,0.172309,0.160821,lama


In [20]:
path_columns = [
    "painting_id",
    "mask_id",
    "mask_type",
    "clean_path_opencv",
    "damaged_path_opencv",
    "mask_path_opencv",
    "restored_path_opencv",
    "restored_path_lama",
]

comparison_unified_df = comparison_unified_df.merge(
    model_pairing_df[path_columns],
    on=["painting_id", "mask_id", "mask_type"],
    how="left",
    validate="one_to_one",
)

comparison_unified_df = comparison_unified_df.rename(
    columns={
        "clean_path_opencv": "clean_path",
        "damaged_path_opencv": "damaged_path",
        "mask_path_opencv": "mask_path",
    }
)

required_path_columns = [
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path_opencv",
    "restored_path_lama",
]

for path_column in required_path_columns:
    if comparison_unified_df[path_column].isna().any():
        raise ValueError(f"Unified comparison contains missing values in {path_column!r}.")

local_winner_columns = [
    "winner_mse_improvement",
    "winner_lpips_improvement",
    "winner_clip_similarity_improvement",
    "winner_dinov2_similarity_improvement",
]

def count_model_wins(row: pd.Series, model: str) -> int:
    return int(sum(row[column] == model for column in local_winner_columns))


def overall_metric_vote(row: pd.Series) -> str:
    lama_wins = row["lama_metric_wins"]
    opencv_wins = row["opencv_metric_wins"]

    if lama_wins > opencv_wins:
        return "lama"

    if opencv_wins > lama_wins:
        return "opencv"

    return "mixed_tie"


comparison_unified_df["lama_metric_wins"] = comparison_unified_df.apply(
    lambda row: count_model_wins(row, "lama"),
    axis=1,
)

comparison_unified_df["opencv_metric_wins"] = comparison_unified_df.apply(
    lambda row: count_model_wins(row, "opencv"),
    axis=1,
)

comparison_unified_df["metric_ties"] = comparison_unified_df.apply(
    lambda row: count_model_wins(row, "tie"),
    axis=1,
)

comparison_unified_df["overall_metric_vote"] = comparison_unified_df.apply(
    overall_metric_vote,
    axis=1,
)

print("Overall metric vote counts:")
display(comparison_unified_df["overall_metric_vote"].value_counts(dropna=False))

print("\nMetric win-count summary:")
display(
    comparison_unified_df[
        [
            "lama_metric_wins",
            "opencv_metric_wins",
            "metric_ties",
            "overall_metric_vote",
        ]
    ].describe()
)

display(
    comparison_unified_df[
        [
            "case_id",
            "painting_id",
            "category",
            "mask_type",
            "winner_mse_improvement",
            "winner_lpips_improvement",
            "winner_clip_similarity_improvement",
            "winner_dinov2_similarity_improvement",
            "lama_metric_wins",
            "opencv_metric_wins",
            "metric_ties",
            "overall_metric_vote",
        ]
    ].head()
)

Overall metric vote counts:


overall_metric_vote
lama         174
opencv        17
mixed_tie      9
Name: count, dtype: int64


Metric win-count summary:


,lama_metric_wins,opencv_metric_wins,metric_ties
count,200.000000,200.000000,200.0
mean,3.475000,0.525000,0.0
std,0.987039,0.987039,0.0
min,0.000000,0.000000,0.0
25%,3.000000,0.000000,0.0
50%,4.000000,0.000000,0.0
75%,4.000000,1.000000,0.0
max,4.000000,4.000000,0.0


,case_id,painting_id,category,mask_type,winner_mse_improvement,winner_lpips_improvement,winner_clip_similarity_improvement,winner_dinov2_similarity_improvement,lama_metric_wins,opencv_metric_wins,metric_ties,overall_metric_vote
0,p001_loss_large,p001,portrait_figure,loss_large,lama,lama,lama,lama,4,0,0,lama
1,p001_loss_small,p001,portrait_figure,loss_small,lama,lama,lama,lama,4,0,0,lama
2,p001_mixed_damage,p001,portrait_figure,mixed_damage,opencv,lama,lama,lama,3,1,0,lama
3,p001_scratch_thin,p001,portrait_figure,scratch_thin,lama,lama,lama,lama,4,0,0,lama
4,p002_loss_large,p002,portrait_figure,loss_large,lama,lama,lama,lama,4,0,0,lama


In [22]:
def resolve_project_path(path_value: str | Path) -> Path:
    """Resolve a stored project path to an absolute filesystem path."""
    path = Path(str(path_value))

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


for path_column in required_path_columns:
    resolved_column = f"{path_column}_resolved"
    comparison_unified_df[resolved_column] = comparison_unified_df[path_column].map(resolve_project_path)

    missing_paths = [
        path
        for path in comparison_unified_df[resolved_column]
        if not path.exists()
    ]

    print(path_column, "missing after PROJECT_ROOT resolution:", len(missing_paths))

    if missing_paths:
        raise FileNotFoundError(
            f"Still missing files after resolving {path_column!r}: "
            f"{missing_paths[:10]}"
        )

print("All unified comparison paths resolve from PROJECT_ROOT.")

clean_path missing after PROJECT_ROOT resolution: 0
damaged_path missing after PROJECT_ROOT resolution: 0
mask_path missing after PROJECT_ROOT resolution: 0
restored_path_opencv missing after PROJECT_ROOT resolution: 0
restored_path_lama missing after PROJECT_ROOT resolution: 0
All unified comparison paths resolve from PROJECT_ROOT.


In [23]:
if len(comparison_unified_df) != 200:
    raise ValueError(
        f"Expected 200 unified local comparison rows, found {len(comparison_unified_df)}."
    )

expected_unified_mask_counts = {
    "loss_large": 50,
    "loss_small": 50,
    "mixed_damage": 50,
    "scratch_thin": 50,
}

actual_unified_mask_counts = comparison_unified_df["mask_type"].value_counts().to_dict()

print("Expected unified mask counts:", expected_unified_mask_counts)
print("Actual unified mask counts:", actual_unified_mask_counts)

for mask_type, expected_count in expected_unified_mask_counts.items():
    actual_count = actual_unified_mask_counts.get(mask_type, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Unified mask type {mask_type!r}: expected {expected_count}, found {actual_count}."
        )

expected_unified_category_counts = {
    "portrait_figure": 40,
    "landscape_natural": 40,
    "architecture_structured": 40,
    "abstraction_surrealism": 40,
    "high_texture_brushwork": 40,
}

actual_unified_category_counts = comparison_unified_df["category"].value_counts().to_dict()

print("\nExpected unified category counts:", expected_unified_category_counts)
print("Actual unified category counts:", actual_unified_category_counts)

for category, expected_count in expected_unified_category_counts.items():
    actual_count = actual_unified_category_counts.get(category, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Unified category {category!r}: expected {expected_count}, found {actual_count}."
        )

required_unified_columns = [
    "mse_improvement_delta_lama_minus_opencv",
    "lpips_improvement_delta_lama_minus_opencv",
    "clip_similarity_improvement_delta_lama_minus_opencv",
    "dinov2_similarity_improvement_delta_lama_minus_opencv",
    "lama_metric_wins",
    "opencv_metric_wins",
    "overall_metric_vote",
]

for column in required_unified_columns:
    if comparison_unified_df[column].isna().any():
        raise ValueError(f"Unified comparison contains missing values in {column!r}.")

for path_column in required_path_columns:
    missing_paths = [
        resolve_project_path(path)
        for path in comparison_unified_df[path_column]
        if str(path).strip() == "" or not resolve_project_path(path).exists()
    ]

    if missing_paths:
        raise FileNotFoundError(
            f"Unified comparison references missing files in {path_column!r}: "
            f"{missing_paths[:10]}"
        )

comparison_unified_df.to_csv(comparison_unified_path, index=False)

print("\nSaved unified comparison:")
print(comparison_unified_path)

print("\nUnified comparison validation passed.")

Expected unified mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}
Actual unified mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}

Expected unified category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}
Actual unified category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}

Saved unified comparison:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_opencv_lama_50.csv

Unified comparison validation passed.


In [24]:
comparison_unified_df["lama_mse_win_but_dinov2_loss"] = (
    comparison_unified_df["winner_mse_improvement"].eq("lama")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("opencv")
)

comparison_unified_df["lama_lpips_win_but_dinov2_loss"] = (
    comparison_unified_df["winner_lpips_improvement"].eq("lama")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("opencv")
)

comparison_unified_df["opencv_mse_loss_but_dinov2_win"] = (
    comparison_unified_df["winner_mse_improvement"].eq("lama")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("opencv")
)

comparison_unified_df["opencv_lpips_loss_but_dinov2_win"] = (
    comparison_unified_df["winner_lpips_improvement"].eq("lama")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("opencv")
)

comparison_unified_df["lama_all_main_metrics_win"] = (
    comparison_unified_df["winner_mse_improvement"].eq("lama")
    & comparison_unified_df["winner_lpips_improvement"].eq("lama")
    & comparison_unified_df["winner_clip_similarity_improvement"].eq("lama")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("lama")
)

comparison_unified_df["opencv_all_main_metrics_win"] = (
    comparison_unified_df["winner_mse_improvement"].eq("opencv")
    & comparison_unified_df["winner_lpips_improvement"].eq("opencv")
    & comparison_unified_df["winner_clip_similarity_improvement"].eq("opencv")
    & comparison_unified_df["winner_dinov2_similarity_improvement"].eq("opencv")
)

comparison_unified_df["mixed_metric_outcome"] = (
    comparison_unified_df["overall_metric_vote"].eq("mixed_tie")
    | (
        comparison_unified_df["lama_metric_wins"].gt(0)
        & comparison_unified_df["opencv_metric_wins"].gt(0)
    )
)

disagreement_flag_columns = [
    "lama_mse_win_but_dinov2_loss",
    "lama_lpips_win_but_dinov2_loss",
    "opencv_mse_loss_but_dinov2_win",
    "opencv_lpips_loss_but_dinov2_win",
    "lama_all_main_metrics_win",
    "opencv_all_main_metrics_win",
    "mixed_metric_outcome",
]

print("Metric disagreement / agreement flag counts:")

flag_summary_df = pd.DataFrame(
    [
        {
            "flag": column,
            "count": int(comparison_unified_df[column].sum()),
            "rate": float(comparison_unified_df[column].mean()),
        }
        for column in disagreement_flag_columns
    ]
)

display(flag_summary_df)

comparison_unified_df.to_csv(comparison_unified_path, index=False)

print("Updated unified comparison with disagreement flags:")
print(comparison_unified_path)

Metric disagreement / agreement flag counts:


,flag,count,rate
0,lama_mse_win_but_dinov2_loss,11,0.055
1,lama_lpips_win_but_dinov2_loss,16,0.080
2,opencv_mse_loss_but_dinov2_win,11,0.055
3,opencv_lpips_loss_but_dinov2_win,16,0.080
4,lama_all_main_metrics_win,142,0.710
5,opencv_all_main_metrics_win,4,0.020
6,mixed_metric_outcome,54,0.270


Updated unified comparison with disagreement flags:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_opencv_lama_50.csv


In [25]:
disagreement_cases_df = comparison_unified_df[
    comparison_unified_df["mixed_metric_outcome"]
    | comparison_unified_df["lama_mse_win_but_dinov2_loss"]
    | comparison_unified_df["lama_lpips_win_but_dinov2_loss"]
    | comparison_unified_df["opencv_mse_loss_but_dinov2_win"]
    | comparison_unified_df["opencv_lpips_loss_but_dinov2_win"]
].copy()

disagreement_cases_df = disagreement_cases_df.sort_values(
    [
        "mixed_metric_outcome",
        "lama_mse_win_but_dinov2_loss",
        "lama_lpips_win_but_dinov2_loss",
        "category",
        "mask_type",
        "case_id",
    ],
    ascending=[False, False, False, True, True, True],
).reset_index(drop=True)

disagreement_cases_df.to_csv(comparison_disagreement_cases_path, index=False)

print("Disagreement cases:", len(disagreement_cases_df))
print("Saved disagreement cases:")
print(comparison_disagreement_cases_path)

display(
    disagreement_cases_df[
        [
            "case_id",
            "painting_id",
            "category",
            "title",
            "mask_type",
            "winner_mse_improvement",
            "winner_lpips_improvement",
            "winner_clip_similarity_improvement",
            "winner_dinov2_similarity_improvement",
            "lama_metric_wins",
            "opencv_metric_wins",
            "metric_ties",
            "overall_metric_vote",
            "lama_mse_win_but_dinov2_loss",
            "lama_lpips_win_but_dinov2_loss",
            "mixed_metric_outcome",
        ]
    ].head(30)
)

Disagreement cases: 54
Saved disagreement cases:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_metric_disagreement_cases_opencv_lama_50.csv


,case_id,painting_id,category,title,mask_type,winner_mse_improvement,winner_lpips_improvement,winner_clip_similarity_improvement,winner_dinov2_similarity_improvement,lama_metric_wins,opencv_metric_wins,metric_ties,overall_metric_vote,lama_mse_win_but_dinov2_loss,lama_lpips_win_but_dinov2_loss,mixed_metric_outcome
0,p039_scratch_thin,p039,abstraction_surrealism,Painting with Troika,scratch_thin,lama,lama,opencv,opencv,2,2,0,mixed_tie,True,True,True
1,p042_scratch_thin,p042,high_texture_brushwork,View of Beirut,scratch_thin,lama,lama,opencv,opencv,2,2,0,mixed_tie,True,True,True
2,p048_scratch_thin,p048,high_texture_brushwork,A Farm in Brittany,scratch_thin,lama,lama,lama,opencv,3,1,0,lama,True,True,True
3,p049_scratch_thin,p049,high_texture_brushwork,Peonies,scratch_thin,lama,lama,opencv,opencv,2,2,0,mixed_tie,True,True,True
4,p012_scratch_thin,p012,landscape_natural,An Extensive Wooded Landscape,scratch_thin,lama,lama,lama,opencv,3,1,0,lama,True,True,True
5,p016_scratch_thin,p016,landscape_natural,Wooded Landscape with a House beside a River,scratch_thin,lama,lama,opencv,opencv,2,2,0,mixed_tie,True,True,True
6,p004_scratch_thin,p004,portrait_figure,Madame Georges Charpentier and Her Children,scratch_thin,lama,lama,opencv,opencv,2,2,0,mixed_tie,True,True,True
7,p008_scratch_thin,p008,portrait_figure,Manuel Osorio Manrique de Zuñiga,scratch_thin,lama,lama,lama,opencv,3,1,0,lama,True,True,True
8,p036_scratch_thin,p036,abstraction_surrealism,"Lozenge Composition with Yellow, Black, Blue, ...",scratch_thin,lama,opencv,opencv,opencv,1,3,0,opencv,True,False,True
9,p023_scratch_thin,p023,architecture_structured,Church Interior,scratch_thin,lama,opencv,opencv,opencv,1,3,0,opencv,True,False,True


In [26]:
def win_rate(series: pd.Series, model: str) -> float:
    return float((series == model).mean())


comparison_summary_by_mask_type_df = (
    comparison_unified_df
    .groupby("mask_type", dropna=False)
    .agg(
        cases=("case_id", "count"),
        mean_mse_delta=("mse_improvement_delta_lama_minus_opencv", "mean"),
        mean_lpips_delta=("lpips_improvement_delta_lama_minus_opencv", "mean"),
        mean_clip_delta=("clip_similarity_improvement_delta_lama_minus_opencv", "mean"),
        mean_dinov2_delta=("dinov2_similarity_improvement_delta_lama_minus_opencv", "mean"),
        lama_mse_win_rate=("winner_mse_improvement", lambda values: win_rate(values, "lama")),
        lama_lpips_win_rate=("winner_lpips_improvement", lambda values: win_rate(values, "lama")),
        lama_clip_win_rate=("winner_clip_similarity_improvement", lambda values: win_rate(values, "lama")),
        lama_dinov2_win_rate=("winner_dinov2_similarity_improvement", lambda values: win_rate(values, "lama")),
        lama_overall_vote_win_rate=("overall_metric_vote", lambda values: win_rate(values, "lama")),
        opencv_overall_vote_win_rate=("overall_metric_vote", lambda values: win_rate(values, "opencv")),
        mixed_vote_rate=("overall_metric_vote", lambda values: win_rate(values, "mixed_tie")),
        mixed_metric_outcome_rate=("mixed_metric_outcome", "mean"),
        lama_mse_win_but_dinov2_loss_rate=("lama_mse_win_but_dinov2_loss", "mean"),
        lama_lpips_win_but_dinov2_loss_rate=("lama_lpips_win_but_dinov2_loss", "mean"),
    )
    .reset_index()
    .round(5)
)

comparison_summary_by_mask_type_df.to_csv(comparison_summary_by_mask_type_path, index=False)

print("Saved comparison summary by mask type:")
print(comparison_summary_by_mask_type_path)

display(comparison_summary_by_mask_type_df)

Saved comparison summary by mask type:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_summary_by_mask_type_opencv_lama_50.csv


,mask_type,cases,mean_mse_delta,mean_lpips_delta,mean_clip_delta,mean_dinov2_delta,lama_mse_win_rate,lama_lpips_win_rate,lama_clip_win_rate,lama_dinov2_win_rate,lama_overall_vote_win_rate,opencv_overall_vote_win_rate,mixed_vote_rate,mixed_metric_outcome_rate,lama_mse_win_but_dinov2_loss_rate,lama_lpips_win_but_dinov2_loss_rate
0,loss_large,50,555.64250,0.12208,0.13188,0.25763,0.84,1.00,1.00,1.00,1.00,0.00,0.00,0.16,0.00,0.00
1,loss_small,50,244.87214,0.01611,0.00489,0.02092,0.80,1.00,0.96,1.00,0.98,0.00,0.02,0.22,0.00,0.00
2,mixed_damage,50,287.27050,0.02606,0.01740,0.05912,0.90,1.00,1.00,1.00,1.00,0.00,0.00,0.10,0.00,0.00
3,scratch_thin,50,67.80452,0.00161,0.00007,-0.00080,0.64,0.76,0.54,0.46,0.50,0.34,0.16,0.60,0.22,0.32


In [27]:
comparison_summary_by_category_df = (
    comparison_unified_df
    .groupby("category", dropna=False)
    .agg(
        cases=("case_id", "count"),
        mean_mse_delta=("mse_improvement_delta_lama_minus_opencv", "mean"),
        mean_lpips_delta=("lpips_improvement_delta_lama_minus_opencv", "mean"),
        mean_clip_delta=("clip_similarity_improvement_delta_lama_minus_opencv", "mean"),
        mean_dinov2_delta=("dinov2_similarity_improvement_delta_lama_minus_opencv", "mean"),
        lama_mse_win_rate=("winner_mse_improvement", lambda values: win_rate(values, "lama")),
        lama_lpips_win_rate=("winner_lpips_improvement", lambda values: win_rate(values, "lama")),
        lama_clip_win_rate=("winner_clip_similarity_improvement", lambda values: win_rate(values, "lama")),
        lama_dinov2_win_rate=("winner_dinov2_similarity_improvement", lambda values: win_rate(values, "lama")),
        lama_overall_vote_win_rate=("overall_metric_vote", lambda values: win_rate(values, "lama")),
        opencv_overall_vote_win_rate=("overall_metric_vote", lambda values: win_rate(values, "opencv")),
        mixed_vote_rate=("overall_metric_vote", lambda values: win_rate(values, "mixed_tie")),
        mixed_metric_outcome_rate=("mixed_metric_outcome", "mean"),
        lama_mse_win_but_dinov2_loss_rate=("lama_mse_win_but_dinov2_loss", "mean"),
        lama_lpips_win_but_dinov2_loss_rate=("lama_lpips_win_but_dinov2_loss", "mean"),
    )
    .reset_index()
    .round(5)
)

comparison_summary_by_category_df.to_csv(comparison_summary_by_category_path, index=False)

print("Saved comparison summary by category:")
print(comparison_summary_by_category_path)

display(comparison_summary_by_category_df)

Saved comparison summary by category:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_summary_by_category_opencv_lama_50.csv


,category,cases,mean_mse_delta,mean_lpips_delta,mean_clip_delta,mean_dinov2_delta,lama_mse_win_rate,lama_lpips_win_rate,lama_clip_win_rate,lama_dinov2_win_rate,lama_overall_vote_win_rate,opencv_overall_vote_win_rate,mixed_vote_rate,mixed_metric_outcome_rate,lama_mse_win_but_dinov2_loss_rate,lama_lpips_win_but_dinov2_loss_rate
0,abstraction_surrealism,40,584.27745,0.02783,0.02267,0.06455,0.825,0.925,0.800,0.825,0.825,0.150,0.025,0.200,0.050,0.100
1,architecture_structured,40,315.81037,0.04869,0.04035,0.06917,0.775,0.825,0.850,0.850,0.800,0.175,0.025,0.225,0.025,0.000
2,high_texture_brushwork,40,169.93699,0.04017,0.04926,0.08200,0.800,1.000,0.875,0.875,0.900,0.025,0.075,0.325,0.075,0.125
3,landscape_natural,40,192.22043,0.05651,0.04799,0.14216,0.725,0.950,0.925,0.850,0.875,0.075,0.050,0.350,0.075,0.100
4,portrait_figure,40,182.24185,0.03411,0.03254,0.06321,0.850,1.000,0.925,0.925,0.950,0.000,0.050,0.250,0.050,0.075


In [28]:
comparison_win_rates_df = pd.DataFrame(
    [
        {
            "comparison_level": "overall",
            "cases": len(comparison_unified_df),
            "lama_mse_win_rate": win_rate(comparison_unified_df["winner_mse_improvement"], "lama"),
            "opencv_mse_win_rate": win_rate(comparison_unified_df["winner_mse_improvement"], "opencv"),
            "mse_tie_rate": win_rate(comparison_unified_df["winner_mse_improvement"], "tie"),
            "lama_lpips_win_rate": win_rate(comparison_unified_df["winner_lpips_improvement"], "lama"),
            "opencv_lpips_win_rate": win_rate(comparison_unified_df["winner_lpips_improvement"], "opencv"),
            "lpips_tie_rate": win_rate(comparison_unified_df["winner_lpips_improvement"], "tie"),
            "lama_clip_win_rate": win_rate(comparison_unified_df["winner_clip_similarity_improvement"], "lama"),
            "opencv_clip_win_rate": win_rate(comparison_unified_df["winner_clip_similarity_improvement"], "opencv"),
            "clip_tie_rate": win_rate(comparison_unified_df["winner_clip_similarity_improvement"], "tie"),
            "lama_dinov2_win_rate": win_rate(comparison_unified_df["winner_dinov2_similarity_improvement"], "lama"),
            "opencv_dinov2_win_rate": win_rate(comparison_unified_df["winner_dinov2_similarity_improvement"], "opencv"),
            "dinov2_tie_rate": win_rate(comparison_unified_df["winner_dinov2_similarity_improvement"], "tie"),
            "lama_overall_vote_win_rate": win_rate(comparison_unified_df["overall_metric_vote"], "lama"),
            "opencv_overall_vote_win_rate": win_rate(comparison_unified_df["overall_metric_vote"], "opencv"),
            "mixed_vote_rate": win_rate(comparison_unified_df["overall_metric_vote"], "mixed_tie"),
            "mixed_metric_outcome_rate": float(comparison_unified_df["mixed_metric_outcome"].mean()),
            "lama_all_main_metrics_win_rate": float(comparison_unified_df["lama_all_main_metrics_win"].mean()),
            "opencv_all_main_metrics_win_rate": float(comparison_unified_df["opencv_all_main_metrics_win"].mean()),
        }
    ]
).round(5)

comparison_win_rates_df.to_csv(comparison_win_rates_path, index=False)

print("Saved overall win-rate summary:")
print(comparison_win_rates_path)

display(comparison_win_rates_df)

Saved overall win-rate summary:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_win_rates_opencv_lama_50.csv


,comparison_level,cases,lama_mse_win_rate,opencv_mse_win_rate,mse_tie_rate,lama_lpips_win_rate,opencv_lpips_win_rate,lpips_tie_rate,lama_clip_win_rate,opencv_clip_win_rate,clip_tie_rate,lama_dinov2_win_rate,opencv_dinov2_win_rate,dinov2_tie_rate,lama_overall_vote_win_rate,opencv_overall_vote_win_rate,mixed_vote_rate,mixed_metric_outcome_rate,lama_all_main_metrics_win_rate,opencv_all_main_metrics_win_rate
0,overall,200,0.795,0.205,0.0,0.94,0.06,0.0,0.875,0.125,0.0,0.865,0.135,0.0,0.87,0.085,0.045,0.27,0.71,0.02


In [29]:
def select_top_difference_cases(
    df: pd.DataFrame,
    *,
    metric_column: str,
    label: str,
    ascending: bool,
    n: int,
) -> pd.DataFrame:
    selected_df = (
        df
        .dropna(subset=[metric_column])
        .sort_values(metric_column, ascending=ascending)
        .head(n)
        .copy()
    )

    selected_df["selection_reason"] = label
    selected_df["selection_metric"] = metric_column

    return selected_df


visual_case_selections = [
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="mse_improvement_delta_lama_minus_opencv",
        label="Strongest LaMa advantage by masked-region MSE improvement",
        ascending=False,
        n=3,
    ),
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="mse_improvement_delta_lama_minus_opencv",
        label="Strongest OpenCV advantage by masked-region MSE improvement",
        ascending=True,
        n=3,
    ),
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="lpips_improvement_delta_lama_minus_opencv",
        label="Strongest LaMa advantage by mask-bbox LPIPS improvement",
        ascending=False,
        n=3,
    ),
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="lpips_improvement_delta_lama_minus_opencv",
        label="Strongest OpenCV advantage by mask-bbox LPIPS improvement",
        ascending=True,
        n=3,
    ),
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="dinov2_similarity_improvement_delta_lama_minus_opencv",
        label="Strongest LaMa advantage by mask-bbox DINOv2 improvement",
        ascending=False,
        n=3,
    ),
    select_top_difference_cases(
        comparison_unified_df,
        metric_column="dinov2_similarity_improvement_delta_lama_minus_opencv",
        label="Strongest OpenCV advantage by mask-bbox DINOv2 improvement",
        ascending=True,
        n=3,
    ),
]

if len(disagreement_cases_df) > 0:
    disagreement_visual_df = (
        disagreement_cases_df
        .sort_values(
            [
                "lama_mse_win_but_dinov2_loss",
                "lama_lpips_win_but_dinov2_loss",
                "mixed_metric_outcome",
                "category",
                "mask_type",
            ],
            ascending=[False, False, False, True, True],
        )
        .head(8)
        .copy()
    )

    disagreement_visual_df["selection_reason"] = "Metric disagreement case"
    disagreement_visual_df["selection_metric"] = "mixed_metric_outcome"
    visual_case_selections.append(disagreement_visual_df)

category_examples_df = (
    comparison_unified_df
    .sort_values("lama_metric_wins", ascending=False)
    .groupby("category", dropna=False)
    .head(1)
    .copy()
)

category_examples_df["selection_reason"] = "Representative category example"
category_examples_df["selection_metric"] = "lama_metric_wins"
visual_case_selections.append(category_examples_df)

visual_cases_raw_df = pd.concat(visual_case_selections, ignore_index=True)

reason_df = (
    visual_cases_raw_df
    .groupby("case_id")
    .agg(
        selection_reason=("selection_reason", lambda values: "; ".join(sorted(set(values)))),
        selection_metric=("selection_metric", lambda values: "; ".join(sorted(set(values)))),
    )
    .reset_index()
)

visual_case_base_columns = [
    column for column in visual_cases_raw_df.columns
    if column not in ["selection_reason", "selection_metric"]
]

comparison_visual_cases_df = (
    visual_cases_raw_df[visual_case_base_columns]
    .drop_duplicates(subset=["case_id"])
    .merge(reason_df, on="case_id", how="left")
    .sort_values(["category", "mask_type", "case_id"])
    .reset_index(drop=True)
)

print("Selected visual comparison cases:", len(comparison_visual_cases_df))

display(
    comparison_visual_cases_df[
        [
            "case_id",
            "painting_id",
            "category",
            "title",
            "mask_type",
            "selection_reason",
            "winner_mse_improvement",
            "winner_lpips_improvement",
            "winner_clip_similarity_improvement",
            "winner_dinov2_similarity_improvement",
            "lama_metric_wins",
            "opencv_metric_wins",
            "overall_metric_vote",
        ]
    ]
)

Selected visual comparison cases: 30


,case_id,painting_id,category,title,mask_type,selection_reason,winner_mse_improvement,winner_lpips_improvement,winner_clip_similarity_improvement,winner_dinov2_similarity_improvement,lama_metric_wins,opencv_metric_wins,overall_metric_vote
0,p031_loss_large,p031,abstraction_surrealism,Improvisation No. 30 (Cannons),loss_large,Representative category example,lama,lama,lama,lama,4,0,lama
1,p032_loss_large,p032,abstraction_surrealism,Composition (No. 1) Gray-Red,loss_large,Strongest LaMa advantage by masked-region MSE ...,lama,lama,lama,lama,4,0,lama
2,p037_loss_large,p037,abstraction_surrealism,In the Sea,loss_large,Strongest OpenCV advantage by masked-region MS...,opencv,lama,lama,lama,3,1,lama
3,p040_loss_large,p040,abstraction_surrealism,Houses at Murnau,loss_large,Strongest OpenCV advantage by masked-region MS...,opencv,lama,lama,lama,3,1,lama
4,p032_loss_small,p032,abstraction_surrealism,Composition (No. 1) Gray-Red,loss_small,Strongest LaMa advantage by masked-region MSE ...,lama,lama,lama,lama,4,0,lama
5,p036_scratch_thin,p036,abstraction_surrealism,"Lozenge Composition with Yellow, Black, Blue, ...",scratch_thin,Strongest OpenCV advantage by mask-bbox DINOv2...,lama,opencv,opencv,opencv,1,3,opencv
6,p039_scratch_thin,p039,abstraction_surrealism,Painting with Troika,scratch_thin,Metric disagreement case,lama,lama,opencv,opencv,2,2,mixed_tie
7,p022_loss_large,p022,architecture_structured,"Interior of a Protestant, Gothic Church, with ...",loss_large,Strongest LaMa advantage by masked-region MSE ...,lama,lama,lama,lama,4,0,lama
8,p029_loss_small,p029,architecture_structured,The Trading Post of the Dutch East India Compa...,loss_small,Representative category example,lama,lama,lama,lama,4,0,lama
9,p024_scratch_thin,p024,architecture_structured,Interior of the Church of St Bavo in Haarlem,scratch_thin,Strongest OpenCV advantage by mask-bbox LPIPS ...,opencv,opencv,opencv,opencv,0,4,opencv


In [30]:
def load_rgb_image(path_value: str | Path) -> Image.Image:
    """Load an image from a project-relative or absolute path as RGB."""
    path = resolve_project_path(path_value)

    if not path.exists():
        raise FileNotFoundError(f"Missing image file: {path}")

    return Image.open(path).convert("RGB")


def load_mask_image(path_value: str | Path) -> Image.Image:
    """Load a mask from a project-relative or absolute path."""
    path = resolve_project_path(path_value)

    if not path.exists():
        raise FileNotFoundError(f"Missing mask file: {path}")

    return Image.open(path).convert("L")


def save_model_comparison_figure(
    row: pd.Series,
    *,
    output_dir: Path,
) -> Path:
    """Save a side-by-side OpenCV versus LaMa comparison figure."""
    clean_img = load_rgb_image(row["clean_path"])
    damaged_img = load_rgb_image(row["damaged_path"])
    mask_img = load_mask_image(row["mask_path"])
    opencv_img = load_rgb_image(row["restored_path_opencv"])
    lama_img = load_rgb_image(row["restored_path_lama"])

    figure_path = output_dir / f"{row['case_id']}_opencv_vs_lama.png"

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    axes[0, 0].imshow(clean_img)
    axes[0, 0].set_title("Clean reference")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(mask_img, cmap="gray")
    axes[0, 1].set_title("Synthetic mask")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(damaged_img)
    axes[0, 2].set_title("Damaged input")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(opencv_img)
    axes[1, 0].set_title("OpenCV Telea restored")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(lama_img)
    axes[1, 1].set_title("LaMa restored")
    axes[1, 1].axis("off")

    metric_text = (
        f"Case: {row['case_id']}\n"
        f"Category: {row['category']}\n"
        f"Mask: {row['mask_type']}\n\n"
        f"MSE winner: {row['winner_mse_improvement']}\n"
        f"LPIPS winner: {row['winner_lpips_improvement']}\n"
        f"CLIP winner: {row['winner_clip_similarity_improvement']}\n"
        f"DINOv2 winner: {row['winner_dinov2_similarity_improvement']}\n\n"
        f"LaMa wins: {row['lama_metric_wins']}\n"
        f"OpenCV wins: {row['opencv_metric_wins']}\n"
        f"Overall vote: {row['overall_metric_vote']}\n\n"
        f"MSE delta: {row['mse_improvement_delta_lama_minus_opencv']:.4f}\n"
        f"LPIPS delta: {row['lpips_improvement_delta_lama_minus_opencv']:.5f}\n"
        f"CLIP delta: {row['clip_similarity_improvement_delta_lama_minus_opencv']:.5f}\n"
        f"DINOv2 delta: {row['dinov2_similarity_improvement_delta_lama_minus_opencv']:.5f}"
    )

    axes[1, 2].text(
        0.02,
        0.98,
        metric_text,
        va="top",
        ha="left",
        fontsize=10,
        family="monospace",
        transform=axes[1, 2].transAxes,
    )
    axes[1, 2].set_title("Metric comparison")
    axes[1, 2].axis("off")

    fig.suptitle(
        f"{row['case_id']} | {row['title']} | {row['selection_reason']}",
        fontsize=12,
    )

    plt.tight_layout()
    fig.savefig(figure_path, dpi=140, bbox_inches="tight")
    plt.close(fig)

    return figure_path


print("Visual comparison helpers ready.")

Visual comparison helpers ready.


In [31]:
figure_paths = []

for index, row in comparison_visual_cases_df.iterrows():
    figure_path = save_model_comparison_figure(
        row,
        output_dir=comparison_figures_dir,
    )

    figure_paths.append(str(figure_path))

    if (index + 1) % 5 == 0 or (index + 1) == len(comparison_visual_cases_df):
        print(
            f"Generated {index + 1}/{len(comparison_visual_cases_df)} "
            f"comparison figures..."
        )

comparison_visual_cases_df["comparison_figure_path"] = figure_paths

comparison_visual_cases_df.to_csv(comparison_visual_cases_path, index=False)

print("\nSaved visual comparison case manifest:")
print(comparison_visual_cases_path)

print("\nGenerated comparison figures:", len(figure_paths))
print("Figure directory:", comparison_figures_dir)

display(
    comparison_visual_cases_df[
        [
            "case_id",
            "category",
            "mask_type",
            "selection_reason",
            "comparison_figure_path",
        ]
    ].head(20)
)

Generated 5/30 comparison figures...
Generated 10/30 comparison figures...
Generated 15/30 comparison figures...
Generated 20/30 comparison figures...
Generated 25/30 comparison figures...
Generated 30/30 comparison figures...

Saved visual comparison case manifest:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_visual_cases_opencv_lama_50.csv

Generated comparison figures: 30
Figure directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\figures\model_comparison\opencv_vs_lama


,case_id,category,mask_type,selection_reason,comparison_figure_path
0,p031_loss_large,abstraction_surrealism,loss_large,Representative category example,D:\Masters\FH\Thesis\painting-restoration-eval...
1,p032_loss_large,abstraction_surrealism,loss_large,Strongest LaMa advantage by masked-region MSE ...,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p037_loss_large,abstraction_surrealism,loss_large,Strongest OpenCV advantage by masked-region MS...,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p040_loss_large,abstraction_surrealism,loss_large,Strongest OpenCV advantage by masked-region MS...,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p032_loss_small,abstraction_surrealism,loss_small,Strongest LaMa advantage by masked-region MSE ...,D:\Masters\FH\Thesis\painting-restoration-eval...
5,p036_scratch_thin,abstraction_surrealism,scratch_thin,Strongest OpenCV advantage by mask-bbox DINOv2...,D:\Masters\FH\Thesis\painting-restoration-eval...
6,p039_scratch_thin,abstraction_surrealism,scratch_thin,Metric disagreement case,D:\Masters\FH\Thesis\painting-restoration-eval...
7,p022_loss_large,architecture_structured,loss_large,Strongest LaMa advantage by masked-region MSE ...,D:\Masters\FH\Thesis\painting-restoration-eval...
8,p029_loss_small,architecture_structured,loss_small,Representative category example,D:\Masters\FH\Thesis\painting-restoration-eval...
9,p024_scratch_thin,architecture_structured,scratch_thin,Strongest OpenCV advantage by mask-bbox LPIPS ...,D:\Masters\FH\Thesis\painting-restoration-eval...


In [32]:
if len(comparison_visual_cases_df) == 0:
    raise ValueError("No visual comparison cases were selected.")

if not comparison_visual_cases_path.exists():
    raise FileNotFoundError(f"Missing visual case manifest: {comparison_visual_cases_path}")

missing_comparison_figures = [
    path
    for path in comparison_visual_cases_df["comparison_figure_path"]
    if not Path(path).exists()
]

if missing_comparison_figures:
    raise FileNotFoundError(
        f"Missing generated comparison figures: {missing_comparison_figures[:10]}"
    )

saved_visual_cases_df = pd.read_csv(comparison_visual_cases_path)

if len(saved_visual_cases_df) != len(comparison_visual_cases_df):
    raise ValueError(
        f"Saved visual case manifest row count mismatch: "
        f"expected {len(comparison_visual_cases_df)}, found {len(saved_visual_cases_df)}."
    )

print("Visual comparison cases:", len(comparison_visual_cases_df))
print("All visual comparison figures exist.")

Visual comparison cases: 30
All visual comparison figures exist.


In [33]:
def build_comparison_overview_table() -> pd.DataFrame:
    """Build a compact overview table for the OpenCV versus LaMa comparison."""
    return pd.DataFrame(
        [
            {"item": "Paintings", "value": processed_metadata_df["painting_id"].nunique()},
            {"item": "Paired restoration cases", "value": len(model_pairing_df)},
            {"item": "Non-zero local comparison cases", "value": len(comparison_unified_df)},
            {"item": "Mask types in local comparison", "value": comparison_unified_df["mask_type"].nunique()},
            {"item": "Painting categories", "value": comparison_unified_df["category"].nunique()},
            {"item": "Compared models", "value": "OpenCV Telea, LaMa"},
            {"item": "Main classical region", "value": "masked_region"},
            {"item": "Main LPIPS / feature region", "value": "mask_bbox_crop"},
        ]
    )


def build_comparison_key_findings_html(
    *,
    summary_by_mask: pd.DataFrame,
    summary_by_category: pd.DataFrame,
    win_rates: pd.DataFrame,
    flag_summary: pd.DataFrame,
) -> str:
    """Build textual key findings for the OpenCV versus LaMa report."""
    overall = win_rates.iloc[0]

    strongest_lama_mask_mse = summary_by_mask.sort_values(
        "mean_mse_delta",
        ascending=False,
    ).iloc[0]

    strongest_opencv_mask_dino = summary_by_mask.sort_values(
        "mean_dinov2_delta",
        ascending=True,
    ).iloc[0]

    strongest_lama_category_vote = summary_by_category.sort_values(
        "lama_overall_vote_win_rate",
        ascending=False,
    ).iloc[0]

    most_mixed_category = summary_by_category.sort_values(
        "mixed_metric_outcome_rate",
        ascending=False,
    ).iloc[0]

    flag_html = dataframe_to_html_table(
        flag_summary,
        float_decimals=4,
    )

    return f"""
    <h2>Key comparison findings</h2>

    <p>
        Across the 200 non-zero local comparison cases, LaMa achieved an overall metric-vote
        win rate of <b>{float(overall['lama_overall_vote_win_rate']):.3f}</b>, while OpenCV Telea
        achieved <b>{float(overall['opencv_overall_vote_win_rate']):.3f}</b>. Mixed or tied
        metric votes occurred at a rate of <b>{float(overall['mixed_vote_rate']):.3f}</b>.
    </p>

    <p>
        The strongest average LaMa advantage by masked-region MSE improvement was observed for
        <b>{html_escape(strongest_lama_mask_mse['mask_type'])}</b>. This indicates where learned
        inpainting most improved local pixel-level recovery compared with the deterministic
        OpenCV baseline.
    </p>

    <p>
        The strongest average OpenCV advantage by DINOv2 feature-space behavior was observed for
        <b>{html_escape(strongest_opencv_mask_dino['mask_type'])}</b>. This is important because
        feature-space behavior can reveal structural differences that are not fully captured by
        pixel-level error reduction.
    </p>

    <p>
        The category with the highest LaMa overall vote win rate was
        <b>{html_escape(strongest_lama_category_vote['category'])}</b>. The category with the
        highest mixed-metric outcome rate was <b>{html_escape(most_mixed_category['category'])}</b>.
        This supports category-aware evaluation rather than treating all paintings as one
        homogeneous image set.
    </p>

    <p>
        Metric disagreement is a central result rather than noise. Cases where one model wins
        MSE or LPIPS but loses DINOv2 indicate that restoration behavior cannot be reduced to a
        single scalar metric.
    </p>

    <h3>Metric disagreement and agreement flags</h3>
    {flag_html}
    """

In [34]:
def build_comparison_visual_case_sections_html(
    visual_cases_df: pd.DataFrame,
    *,
    project_root: Path,
    image_mode: str = "embedded",
    image_width: int = 980,
) -> str:
    """Build HTML sections for selected OpenCV versus LaMa comparison cases."""
    if visual_cases_df.empty:
        return "<p>No visual comparison cases were selected.</p>"

    sections = []

    for _, row in visual_cases_df.iterrows():
        figure_path_value = row.get("comparison_figure_path", "")

        if pd.notna(figure_path_value) and str(figure_path_value).strip() != "":
            figure_html = image_block(
                Path(str(figure_path_value)),
                caption="OpenCV Telea versus LaMa visual comparison",
                project_root=project_root,
                width=image_width,
                mode=image_mode,
            )
        else:
            figure_html = """
            <p class="missing">
                No comparison figure path available for this case.
            </p>
            """

        metric_table = pd.DataFrame(
            [
                {
                    "case_id": row.get("case_id", ""),
                    "category": row.get("category", ""),
                    "mask_type": row.get("mask_type", ""),
                    "mse_delta": row.get("mse_improvement_delta_lama_minus_opencv", pd.NA),
                    "lpips_delta": row.get("lpips_improvement_delta_lama_minus_opencv", pd.NA),
                    "clip_delta": row.get("clip_similarity_improvement_delta_lama_minus_opencv", pd.NA),
                    "dinov2_delta": row.get("dinov2_similarity_improvement_delta_lama_minus_opencv", pd.NA),
                    "lama_wins": row.get("lama_metric_wins", pd.NA),
                    "opencv_wins": row.get("opencv_metric_wins", pd.NA),
                    "overall_vote": row.get("overall_metric_vote", ""),
                }
            ]
        )

        sections.append(
            f"""
            <section class="case-section">
                <h3>{html_escape(row.get("case_id", ""))}: {html_escape(row.get("title", ""))}</h3>

                <p>
                    <b>Selection reason:</b> {html_escape(row.get("selection_reason", ""))}<br>
                    <b>Painting ID:</b> {html_escape(row.get("painting_id", ""))}<br>
                    <b>Category:</b> {html_escape(row.get("category", ""))}<br>
                    <b>Mask type:</b> {html_escape(row.get("mask_type", ""))}<br>
                    <b>Artist:</b> {html_escape(row.get("artist", ""))}<br>
                    <b>Date:</b> {html_escape(row.get("date", ""))}
                </p>

                {dataframe_to_html_table(metric_table, float_decimals=5)}

                <p>
                    Positive deltas mean LaMa is better than OpenCV for improvement-based metrics.
                    Negative deltas mean OpenCV is better. The overall vote is a compact diagnostic
                    signal, not a final conservation judgment.
                </p>

                {figure_html}
            </section>
            """
        )

    return "\n".join(sections)

In [35]:
def build_opencv_lama_comparison_report_html(
    *,
    overview_df: pd.DataFrame,
    summary_by_mask: pd.DataFrame,
    summary_by_category: pd.DataFrame,
    win_rates: pd.DataFrame,
    flag_summary: pd.DataFrame,
    visual_cases_df: pd.DataFrame,
    project_root: Path,
    image_mode: str = "embedded",
    title: str = "OpenCV Telea versus LaMa 50-Painting Comparison Report",
) -> str:
    """Build the full OpenCV versus LaMa HTML comparison report."""
    overview_html = dataframe_to_html_table(overview_df, float_decimals=4)
    mask_summary_html = dataframe_to_html_table(summary_by_mask, float_decimals=5)
    category_summary_html = dataframe_to_html_table(summary_by_category, float_decimals=5)
    win_rates_html = dataframe_to_html_table(win_rates, float_decimals=5)

    key_findings_html = build_comparison_key_findings_html(
        summary_by_mask=summary_by_mask,
        summary_by_category=summary_by_category,
        win_rates=win_rates,
        flag_summary=flag_summary,
    )

    visual_cases_html = build_comparison_visual_case_sections_html(
        visual_cases_df,
        project_root=project_root,
        image_mode=image_mode,
    )

    return f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>{html_escape(title)}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: #f7f7f7;
                color: #222;
                line-height: 1.55;
            }}

            h1 {{
                color: #111;
                border-bottom: 2px solid #333;
                padding-bottom: 10px;
            }}

            h2 {{
                margin-top: 38px;
                color: #222;
            }}

            h3 {{
                margin-top: 26px;
                color: #333;
            }}

            .summary,
            .case-section {{
                background: white;
                padding: 22px;
                border-radius: 8px;
                margin-bottom: 30px;
                border: 1px solid #ddd;
            }}

            .case-section {{
                page-break-inside: avoid;
            }}

            .image-block {{
                text-align: center;
                font-size: 12px;
                margin-top: 15px;
            }}

            .image-block img {{
                border: 1px solid #ccc;
                background: #eee;
                max-width: 100%;
                height: auto;
            }}

            .caption {{
                margin-top: 6px;
                color: #555;
            }}

            .missing {{
                color: #9a3412;
                font-style: italic;
            }}

            table,
            .summary-table {{
                border-collapse: collapse;
                margin-top: 15px;
                margin-bottom: 18px;
                width: 100%;
                font-size: 13px;
                background: white;
            }}

            th,
            td,
            .summary-table th,
            .summary-table td {{
                border: 1px solid #ccc;
                padding: 7px;
                text-align: center;
                vertical-align: middle;
            }}

            th,
            .summary-table th {{
                background-color: #eee;
                font-weight: bold;
            }}

            .note {{
                background: #fff7ed;
                border: 1px solid #fed7aa;
                padding: 12px;
                border-radius: 6px;
            }}
        </style>
    </head>

    <body>
        <h1>{html_escape(title)}</h1>

        <div class="summary">
            <h2>Experiment overview</h2>

            <p>
                This report compares the deterministic OpenCV Telea baseline with the pretrained
                LaMa inpainting baseline on the controlled 50-painting subset.
            </p>

            <p>
                The comparison is case-paired: both models are evaluated on the same paintings,
                masks, damaged inputs, and clean references. The main local comparison uses
                masked-region classical metrics and mask-bounding-box LPIPS, CLIP, and DINOv2
                metrics.
            </p>

            {overview_html}

            <p class="note">
                The overall metric vote is a diagnostic summary across MSE, LPIPS, CLIP, and DINOv2.
                It should not be interpreted as a conservation-level truth label. Its purpose is to
                identify where metric families agree, disagree, or expose different failure modes.
            </p>
        </div>

        <div class="summary">
            <h2>Overall win-rate summary</h2>
            {win_rates_html}

            <h2>Summary by mask type</h2>
            {mask_summary_html}

            <h2>Summary by painting category</h2>
            {category_summary_html}

            {key_findings_html}
        </div>

        <div class="summary">
            <h2>Selected visual comparison cases</h2>

            <p>
                The following cases were selected from strongest model advantages, feature-space
                differences, metric-disagreement cases, and category examples. These figures are
                intended to support qualitative inspection of the metric results.
            </p>

            {visual_cases_html}
        </div>

        <div class="summary">
            <h2>Comparison conclusion</h2>

            <p>
                OpenCV Telea and LaMa represent different restoration paradigms: deterministic
                local interpolation versus learned pretrained inpainting. Comparing them on the
                same controlled damage cases shows where learned inpainting provides measurable
                local advantages and where classical interpolation remains competitive or more
                structurally conservative.
            </p>

            <p>
                The most important result is not simply which model wins more cases. The more
                thesis-relevant finding is that metric families can disagree. Pixel-level,
                perceptual, feature-space, and visual diagnostic evidence each reveal different
                aspects of restoration behavior.
            </p>

            <p>
                This comparison therefore strengthens the evaluation-framework argument and prepares
                the project for the next model family: diffusion-based inpainting.
            </p>
        </div>
    </body>
    </html>
    """

In [36]:
comparison_overview_df = build_comparison_overview_table()

comparison_report_html = build_opencv_lama_comparison_report_html(
    overview_df=comparison_overview_df,
    summary_by_mask=comparison_summary_by_mask_type_df,
    summary_by_category=comparison_summary_by_category_df,
    win_rates=comparison_win_rates_df,
    flag_summary=flag_summary_df,
    visual_cases_df=comparison_visual_cases_df,
    project_root=PROJECT_ROOT,
    image_mode="embedded",
    title="OpenCV Telea versus LaMa 50-Painting Comparison Report",
)

comparison_report_path.parent.mkdir(parents=True, exist_ok=True)
comparison_report_path.write_text(comparison_report_html, encoding="utf-8")

print("Saved OpenCV versus LaMa comparison report:")
print(comparison_report_path)

print("Report size in characters:", len(comparison_report_html))
print("Report size in bytes:", comparison_report_path.stat().st_size)

Saved OpenCV versus LaMa comparison report:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_vs_lama_comparison_report_50.html
Report size in characters: 65736037
Report size in bytes: 65738349


In [37]:
expected_comparison_files = [
    model_pairing_path,
    comparison_classical_path,
    comparison_lpips_path,
    comparison_feature_path,
    comparison_unified_path,
    comparison_summary_by_mask_type_path,
    comparison_summary_by_category_path,
    comparison_win_rates_path,
    comparison_disagreement_cases_path,
    comparison_visual_cases_path,
    comparison_report_path,
]

for output_file in expected_comparison_files:
    if not output_file.exists():
        raise FileNotFoundError(f"Missing expected comparison output: {output_file}")

saved_pairing_df = pd.read_csv(model_pairing_path)
saved_classical_df = pd.read_csv(comparison_classical_path)
saved_lpips_df = pd.read_csv(comparison_lpips_path)
saved_feature_df = pd.read_csv(comparison_feature_path)
saved_unified_df = pd.read_csv(comparison_unified_path)
saved_mask_summary_df = pd.read_csv(comparison_summary_by_mask_type_path)
saved_category_summary_df = pd.read_csv(comparison_summary_by_category_path)
saved_win_rates_df = pd.read_csv(comparison_win_rates_path)
saved_disagreement_df = pd.read_csv(comparison_disagreement_cases_path)
saved_visual_cases_df = pd.read_csv(comparison_visual_cases_path)

expected_saved_row_counts = {
    "pairing": 250,
    "classical": 900,
    "lpips": 700,
    "feature": 700,
    "unified": 200,
    "mask_summary": 4,
    "category_summary": 5,
    "win_rates": 1,
}

actual_saved_row_counts = {
    "pairing": len(saved_pairing_df),
    "classical": len(saved_classical_df),
    "lpips": len(saved_lpips_df),
    "feature": len(saved_feature_df),
    "unified": len(saved_unified_df),
    "mask_summary": len(saved_mask_summary_df),
    "category_summary": len(saved_category_summary_df),
    "win_rates": len(saved_win_rates_df),
}

print("Expected saved row counts:")
display(pd.Series(expected_saved_row_counts, name="expected"))

print("Actual saved row counts:")
display(pd.Series(actual_saved_row_counts, name="actual"))

for name, expected_count in expected_saved_row_counts.items():
    actual_count = actual_saved_row_counts[name]

    if actual_count != expected_count:
        raise ValueError(
            f"{name}: expected {expected_count} rows, found {actual_count}."
        )

expected_unified_mask_counts = {
    "loss_large": 50,
    "loss_small": 50,
    "mixed_damage": 50,
    "scratch_thin": 50,
}

actual_unified_mask_counts = saved_unified_df["mask_type"].value_counts().to_dict()

for mask_type, expected_count in expected_unified_mask_counts.items():
    actual_count = actual_unified_mask_counts.get(mask_type, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Saved unified mask type {mask_type!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

expected_unified_category_counts = {
    "portrait_figure": 40,
    "landscape_natural": 40,
    "architecture_structured": 40,
    "abstraction_surrealism": 40,
    "high_texture_brushwork": 40,
}

actual_unified_category_counts = saved_unified_df["category"].value_counts().to_dict()

for category, expected_count in expected_unified_category_counts.items():
    actual_count = actual_unified_category_counts.get(category, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Saved unified category {category!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

required_unified_columns = [
    "winner_mse_improvement",
    "winner_lpips_improvement",
    "winner_clip_similarity_improvement",
    "winner_dinov2_similarity_improvement",
    "lama_metric_wins",
    "opencv_metric_wins",
    "metric_ties",
    "overall_metric_vote",
    "mixed_metric_outcome",
]

for column in required_unified_columns:
    if column not in saved_unified_df.columns:
        raise ValueError(f"Saved unified comparison missing required column: {column}")

    if saved_unified_df[column].isna().any():
        raise ValueError(f"Saved unified comparison contains missing values in {column!r}.")

if len(saved_visual_cases_df) == 0:
    raise ValueError("Saved visual comparison case manifest is empty.")

missing_visual_figures = [
    path
    for path in saved_visual_cases_df["comparison_figure_path"]
    if str(path).strip() == "" or not Path(path).exists()
]

if missing_visual_figures:
    raise FileNotFoundError(
        f"Saved visual case manifest references missing figures: "
        f"{missing_visual_figures[:10]}"
    )

report_text = comparison_report_path.read_text(encoding="utf-8")

required_report_phrases = [
    "OpenCV Telea versus LaMa 50-Painting Comparison Report",
    "Experiment overview",
    "Overall win-rate summary",
    "Summary by mask type",
    "Summary by painting category",
    "Metric disagreement and agreement flags",
    "Selected visual comparison cases",
    "Comparison conclusion",
]

missing_report_phrases = [
    phrase for phrase in required_report_phrases
    if phrase not in report_text
]

if missing_report_phrases:
    raise ValueError(
        f"Comparison HTML report missing expected phrases: {missing_report_phrases}"
    )

print("Saved visual comparison cases:", len(saved_visual_cases_df))
print("Saved disagreement cases:", len(saved_disagreement_df))
print("Comparison report size:", comparison_report_path.stat().st_size, "bytes")
print("Final OpenCV versus LaMa comparison output gates passed.")

Expected saved row counts:


pairing             250
classical           900
lpips               700
feature             700
unified             200
mask_summary          4
category_summary      5
win_rates             1
Name: expected, dtype: int64

Actual saved row counts:


pairing             250
classical           900
lpips               700
feature             700
unified             200
mask_summary          4
category_summary      5
win_rates             1
Name: actual, dtype: int64

Saved visual comparison cases: 30
Saved disagreement cases: 54
Comparison report size: 65738349 bytes
Final OpenCV versus LaMa comparison output gates passed.


## Notebook 17 summary

This notebook compared OpenCV Telea and LaMa on the controlled 50-painting subset.

The comparison was case-paired using:

- `painting_id`,
- `mask_id`,
- `mask_type`.

The notebook generated aligned comparison outputs for:

- classical metrics,
- LPIPS perceptual metrics,
- CLIP and DINOv2 feature-space metrics.

The main local comparison table contains 200 non-zero damage cases and combines:

- masked-region classical metrics,
- mask-bounding-box LPIPS metrics,
- mask-bounding-box CLIP/DINOv2 metrics.

Main outputs:

- `outputs/metrics/model_pairing_opencv_lama_50.csv`
- `outputs/metrics/comparison_classical_opencv_lama_50.csv`
- `outputs/metrics/comparison_lpips_opencv_lama_50.csv`
- `outputs/metrics/comparison_feature_similarity_opencv_lama_50.csv`
- `outputs/metrics/comparison_unified_opencv_lama_50.csv`
- `outputs/metrics/comparison_summary_by_mask_type_opencv_lama_50.csv`
- `outputs/metrics/comparison_summary_by_category_opencv_lama_50.csv`
- `outputs/metrics/comparison_win_rates_opencv_lama_50.csv`
- `outputs/metrics/comparison_metric_disagreement_cases_opencv_lama_50.csv`
- `outputs/metrics/comparison_visual_cases_opencv_lama_50.csv`
- `outputs/figures/model_comparison/opencv_vs_lama/`
- `outputs/reports/opencv_vs_lama_comparison_report_50.html`

The comparison report uses embedded visual figures so that it can be opened as a standalone HTML artifact.

This stage establishes the first direct model-family comparison in the evaluation framework: deterministic local interpolation versus learned pretrained inpainting. The next model-family extension is diffusion-based inpainting.